In [1]:

import feedparser
import pandas as pd
from datetime import datetime, timezone

TEST_FEEDS = [
    ("donga_politics", "https://rss.donga.com/politics.xml"),
    ("khan_politics",  "https://www.khan.co.kr/rss/rssdata/politic_news.xml"),
]

In [2]:
def fetch_feed(name: str, url: str, limit: int = 20) -> pd.DataFrame:
    d = feedparser.parse(url)
    fetched_at = datetime.now(timezone.utc).isoformat()

    rows = []
    for e in d.entries[:limit]:
        rows.append({
            "feed_name": name,
            "feed_url": url,
            "title": getattr(e, "title", None),
            "link": getattr(e, "link", None),
            "published": getattr(e, "published", None),
            "summary": getattr(e, "summary", None),
            "guid": getattr(e, "id", None) or getattr(e, "guid", None),
            "fetched_at": fetched_at,
        })

    return pd.DataFrame(rows)

In [3]:
dfs = []
for name, url in TEST_FEEDS:
    df = fetch_feed(name, url, limit=10)
    print(name, "rows:", len(df))
    dfs.append(df)

out = pd.concat(dfs, ignore_index=True)
out.head(10)

donga_politics rows: 10
khan_politics rows: 10


,feed_name,feed_url,title,link,published,summary,guid,fetched_at
0,donga_politics,https://rss.donga.com/politics.xml,"李대통령 “농협, 불투명한 의사결정 구조 등 개선에 속도내야”",https://www.donga.com/news/Politics/article/al...,"Thu, 14 May 2026 17:55:47 +0900","<img align=""left"" hspace=""10"" src=""https://dim...",None,2026-05-14T09:06:16.554108+00:00
1,donga_politics,https://rss.donga.com/politics.xml,하정우 “북구 1인당 지역총생산 1억2000만원” 발언에…한동훈-박민식 비판,https://www.donga.com/news/Politics/article/al...,"Thu, 14 May 2026 17:48:00 +0900","<img align=""left"" hspace=""10"" src=""https://dim...",None,2026-05-14T09:06:16.554108+00:00
2,donga_politics,https://rss.donga.com/politics.xml,조국당·민주당 김상욱으로 울산시장 후보 단일화…진보당과는 협의 중,https://www.donga.com/news/Politics/article/al...,"Thu, 14 May 2026 17:42:00 +0900","<img align=""left"" hspace=""10"" src=""https://dim...",None,2026-05-14T09:06:16.554108+00:00
3,donga_politics,https://rss.donga.com/politics.xml,[정치 한 컷]오랜만에 보는 ‘여야 웃음’,https://www.donga.com/news/Politics/article/al...,"Thu, 14 May 2026 17:25:00 +0900","<img align=""left"" hspace=""10"" src=""https://dim...",None,2026-05-14T09:06:16.554108+00:00
4,donga_politics,https://rss.donga.com/politics.xml,中 “트럼프와 한반도 문제도 논의”…김정은 메시지 전달했나?,https://www.donga.com/news/Politics/article/al...,"Thu, 14 May 2026 17:21:00 +0900","<img align=""left"" hspace=""10"" src=""https://dim...",None,2026-05-14T09:06:16.554108+00:00
5,donga_politics,https://rss.donga.com/politics.xml,"“요격 성공률 96%인 천궁-2, 전원 공급 없어도 10분내 발사”",https://www.donga.com/news/Politics/article/al...,"Thu, 14 May 2026 17:17:00 +0900","<img align=""left"" hspace=""10"" src=""https://dim...",None,2026-05-14T09:06:16.554108+00:00
6,donga_politics,https://rss.donga.com/politics.xml,새마을운동중앙회 찾은 李 “봉사활동 가장 잘하는 단체일 것”,https://www.donga.com/news/Politics/article/al...,"Thu, 14 May 2026 17:08:00 +0900","<img align=""left"" hspace=""10"" src=""https://dim...",None,2026-05-14T09:06:16.554108+00:00
7,donga_politics,https://rss.donga.com/politics.xml,"조국당 “김용남 투기 의혹, 민주당도 2022년에 지적” 공세",https://www.donga.com/news/Politics/article/al...,"Thu, 14 May 2026 17:06:00 +0900","<img align=""left"" hspace=""10"" src=""https://dim...",None,2026-05-14T09:06:16.554108+00:00
8,donga_politics,https://rss.donga.com/politics.xml,"외교부 고위급 “나무호 공격 주체, 이란 아닐 가능성 낮아”",https://www.donga.com/news/Politics/article/al...,"Thu, 14 May 2026 16:45:00 +0900","<img align=""left"" hspace=""10"" src=""https://dim...",None,2026-05-14T09:06:16.554108+00:00
9,donga_politics,https://rss.donga.com/politics.xml,"李, 새마을운동중앙회 찾아 “박정희때 큰 성과…지금도 유용”",https://www.donga.com/news/Politics/article/al...,"Thu, 14 May 2026 16:40:00 +0900","<img align=""left"" hspace=""10"" src=""https://dim...",None,2026-05-14T09:06:16.554108+00:00


In [4]:
out["link"].nunique(), len(out)

(20, 20)

In [5]:
import hashlib
import re
from urllib.parse import urlparse, urlunparse

def normalize_url(u: str | None) -> str | None:
    if not u:
        return None
    try:
        p = urlparse(u)
        # query/fragment 제거(추적 파라미터로 인한 중복 방지)
        return urlunparse((p.scheme, p.netloc, p.path, "", "", ""))
    except Exception:
        return u

def make_item_id(feed_url: str | None, guid: str | None, link: str | None, title: str | None, published: str | None) -> str:
    base = guid or normalize_url(link) or f"{title}|{published}|{feed_url}"
    base = (base or "").strip()
    return hashlib.sha256(base.encode("utf-8")).hexdigest()

out2 = out.copy()
out2["link_norm"] = out2["link"].map(normalize_url)
out2["item_id"] = out2.apply(lambda r: make_item_id(r["feed_url"], r["guid"], r["link_norm"], r["title"], r["published"]), axis=1)

out2[["feed_name","title","link","link_norm","guid","item_id"]].head(3)

,feed_name,title,link,link_norm,guid,item_id
0,donga_politics,"李대통령 “농협, 불투명한 의사결정 구조 등 개선에 속도내야”",https://www.donga.com/news/Politics/article/al...,https://www.donga.com/news/Politics/article/al...,None,991158f5c2c4353b7260c8a238f0e3f923da8fab38c98b...
1,donga_politics,하정우 “북구 1인당 지역총생산 1억2000만원” 발언에…한동훈-박민식 비판,https://www.donga.com/news/Politics/article/al...,https://www.donga.com/news/Politics/article/al...,None,a6e558d861dc437ca5e31737f98e7963fa81407b3c4502...
2,donga_politics,조국당·민주당 김상욱으로 울산시장 후보 단일화…진보당과는 협의 중,https://www.donga.com/news/Politics/article/al...,https://www.donga.com/news/Politics/article/al...,None,ab1dd7031fbf819a4a9862df960de08d3708f2297f6141...


In [6]:
import sqlite3
from pathlib import Path

DB_PATH = Path("../db/news.db")  # notebooks/ 기준이므로 상위로
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS items (
    item_id TEXT PRIMARY KEY,
    feed_name TEXT,
    feed_url TEXT,
    title TEXT,
    link TEXT,
    link_norm TEXT,
    published TEXT,
    summary TEXT,
    guid TEXT,
    fetched_at TEXT
)
""")

conn.commit()
DB_PATH.as_posix()

'../db/news.db'

In [7]:
records = out2[[
    "item_id","feed_name","feed_url","title","link","link_norm","published","summary","guid","fetched_at"
]].to_records(index=False)

cur.executemany("""
INSERT OR IGNORE INTO items
(item_id, feed_name, feed_url, title, link, link_norm, published, summary, guid, fetched_at)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", list(records))

conn.commit()

cur.execute("SELECT COUNT(*) FROM items")
cur.fetchone()

(60,)

In [8]:
# 동일 데이터 재삽입 시도
cur.executemany("""
INSERT OR IGNORE INTO items
(item_id, feed_name, feed_url, title, link, link_norm, published, summary, guid, fetched_at)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", list(records))
conn.commit()

cur.execute("SELECT COUNT(*) FROM items")
cur.fetchone()

(60,)

In [9]:
from pathlib import Path
import pandas as pd

feeds = [
    # 동아일보
    {"publisher":"donga", "section":"politics",       "feed_name":"donga_politics",       "feed_url":"https://rss.donga.com/politics.xml",       "politics_bucket":""},
    {"publisher":"donga", "section":"economy",        "feed_name":"donga_economy",        "feed_url":"https://rss.donga.com/economy.xml",        "politics_bucket":""},
    {"publisher":"donga", "section":"society",        "feed_name":"donga_society",        "feed_url":"https://rss.donga.com/national.xml",       "politics_bucket":""},
    {"publisher":"donga", "section":"international",  "feed_name":"donga_international",  "feed_url":"https://rss.donga.com/international.xml",  "politics_bucket":""},

    # 경향신문
    {"publisher":"khan",  "section":"politics",       "feed_name":"khan_politics",        "feed_url":"https://www.khan.co.kr/rss/rssdata/politic_news.xml",  "politics_bucket":""},
    {"publisher":"khan",  "section":"economy",        "feed_name":"khan_economy",         "feed_url":"https://www.khan.co.kr/rss/rssdata/economy_news.xml",  "politics_bucket":""},
    {"publisher":"khan",  "section":"society",        "feed_name":"khan_society",         "feed_url":"https://www.khan.co.kr/rss/rssdata/society_news.xml",  "politics_bucket":""},
    {"publisher":"khan",  "section":"international",  "feed_name":"khan_international",   "feed_url":"https://www.khan.co.kr/rss/rssdata/kh_world.xml",      "politics_bucket":""},
]

feeds_df = pd.DataFrame(feeds)

FEEDS_CSV = Path("../configs/feeds.csv")  # notebooks/ 기준 상위 configs/
FEEDS_CSV.parent.mkdir(parents=True, exist_ok=True)
feeds_df.to_csv(FEEDS_CSV, index=False, encoding="utf-8-sig")

print("saved:", FEEDS_CSV.resolve())
print("rows:", len(feeds_df))
feeds_df

saved: /home/epistachio/workspace/news-briefing/data/configs/feeds.csv
rows: 8


,publisher,section,feed_name,feed_url,politics_bucket
0,donga,politics,donga_politics,https://rss.donga.com/politics.xml,
1,donga,economy,donga_economy,https://rss.donga.com/economy.xml,
2,donga,society,donga_society,https://rss.donga.com/national.xml,
3,donga,international,donga_international,https://rss.donga.com/international.xml,
4,khan,politics,khan_politics,https://www.khan.co.kr/rss/rssdata/politic_new...,
5,khan,economy,khan_economy,https://www.khan.co.kr/rss/rssdata/economy_new...,
6,khan,society,khan_society,https://www.khan.co.kr/rss/rssdata/society_new...,
7,khan,international,khan_international,https://www.khan.co.kr/rss/rssdata/kh_world.xml,


In [10]:
from pathlib import Path
import sqlite3
import pandas as pd
import feedparser
from datetime import datetime, timezone
import hashlib
from urllib.parse import urlparse, urlunparse
from tqdm import tqdm

# ---------- (1) 프로젝트 루트 탐색 ----------
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "src").exists() and (p / "notebooks").exists() and (p / "db").exists():
            return p
    # fallback: 현재 위치
    return start

ROOT = find_project_root(Path.cwd())
ROOT

# ---------- (2) feeds.csv 자동 탐색 ----------
CANDIDATES = [
    ROOT / "configs" / "feeds.csv",
    ROOT / "data" / "configs" / "feeds.csv",
    Path("/home/epistachio/workspace/news-briefing/configs/feeds.csv"),
    Path("/home/epistachio/workspace/news-briefing/data/configs/feeds.csv"),
]

FEEDS_CSV = next((p for p in CANDIDATES if p.exists()), None)
if FEEDS_CSV is None:
    raise FileNotFoundError(f"feeds.csv not found. tried: {CANDIDATES}")

feeds_df = pd.read_csv(FEEDS_CSV)
print("feeds.csv:", FEEDS_CSV)
print("feeds rows:", len(feeds_df))
feeds_df.head()

# ---------- (3) URL 정규화 + item_id ----------
def normalize_url(u: str | None) -> str | None:
    if not isinstance(u, str) or not u.strip():
        return None
    p = urlparse(u.strip())
    return urlunparse((p.scheme, p.netloc, p.path, "", "", ""))  # query/fragment 제거

def make_item_id(feed_url: str | None, guid: str | None, link: str | None, title: str | None, published: str | None) -> str:
    base = guid or normalize_url(link) or f"{title}|{published}|{feed_url}"
    base = (base or "").strip()
    return hashlib.sha256(base.encode("utf-8")).hexdigest()

def fetch_feed(feed_name: str, feed_url: str, limit: int = 50) -> pd.DataFrame:
    d = feedparser.parse(feed_url)
    fetched_at = datetime.now(timezone.utc).isoformat()

    rows = []
    for e in d.entries[:limit]:
        link = getattr(e, "link", None)
        guid = getattr(e, "id", None) or getattr(e, "guid", None)
        title = getattr(e, "title", None)
        published = getattr(e, "published", None)
        summary = getattr(e, "summary", None)

        link_norm = normalize_url(link)
        item_id = make_item_id(feed_url, guid, link_norm, title, published)

        rows.append({
            "item_id": item_id,
            "feed_name": feed_name,
            "feed_url": feed_url,
            "title": title,
            "link": link,
            "link_norm": link_norm,
            "published": published,
            "summary": summary,
            "guid": guid,
            "fetched_at": fetched_at,
        })

    return pd.DataFrame(rows)

# ---------- (4) SQLite 적재 ----------
DB_PATH = ROOT / "db" / "news.db"
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS items (
    item_id TEXT PRIMARY KEY,
    feed_name TEXT,
    feed_url TEXT,
    title TEXT,
    link TEXT,
    link_norm TEXT,
    published TEXT,
    summary TEXT,
    guid TEXT,
    fetched_at TEXT
)
""")
conn.commit()

before = cur.execute("SELECT COUNT(*) FROM items").fetchone()[0]

report = []
for _, r in tqdm(feeds_df.iterrows(), total=len(feeds_df)):
    fn = r["feed_name"]
    url = r["feed_url"]

    try:
        df = fetch_feed(fn, url, limit=50)
        fetched = len(df)

        if fetched == 0:
            report.append({"feed_name": fn, "fetched": 0, "inserted": 0, "status": "empty"})
            continue

        records = df[[
            "item_id","feed_name","feed_url","title","link","link_norm","published","summary","guid","fetched_at"
        ]].to_records(index=False)

        cur.executemany("""
        INSERT OR IGNORE INTO items
        (item_id, feed_name, feed_url, title, link, link_norm, published, summary, guid, fetched_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, list(records))
        conn.commit()

        # inserted = 변화량(간접 계산)
        after_tmp = cur.execute("SELECT COUNT(*) FROM items").fetchone()[0]
        inserted = after_tmp - (before + sum(x["inserted"] for x in report))

        report.append({"feed_name": fn, "fetched": fetched, "inserted": inserted, "status": "ok"})

    except Exception as e:
        report.append({"feed_name": fn, "fetched": 0, "inserted": 0, "status": f"error: {type(e).__name__}"})

after = cur.execute("SELECT COUNT(*) FROM items").fetchone()[0]

rep_df = pd.DataFrame(report)
print("DB:", DB_PATH)
print("items before:", before, "after:", after, "delta:", after - before)
rep_df

feeds.csv: /home/epistachio/workspace/news-briefing/configs/feeds.csv
feeds rows: 12


100%|██████████████████████████████████████████████████████| 12/12 [00:01<00:00,  6.67it/s]

DB: /home/epistachio/workspace/news-briefing/db/news.db
items before: 1403 after: 1960 delta: 557


,feed_name,fetched,inserted,status
0,donga_politics,50,50,ok
1,donga_economy,50,49,ok
2,donga_society,50,42,ok
3,donga_international,50,43,ok
4,khan_politics,50,44,ok
5,khan_economy,50,50,ok
6,khan_society,50,50,ok
7,khan_international,50,46,ok
8,newsis_politics,50,50,ok
9,newsis_economy,50,49,ok


In [11]:
from pathlib import Path
import sqlite3
import pandas as pd
from datetime import datetime, timedelta, timezone

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "src").exists() and (p / "notebooks").exists() and (p / "db").exists():
            return p
    return start

ROOT = find_project_root(Path.cwd())
DB_PATH = ROOT / "db" / "news.db"

# feeds.csv 위치(너는 data/configs 쪽에 있음)
FEEDS_CANDIDATES = [ROOT/"configs/feeds.csv", ROOT/"data/configs/feeds.csv"]
FEEDS_CSV = next((p for p in FEEDS_CANDIDATES if p.exists()), None)
if FEEDS_CSV is None:
    raise FileNotFoundError(f"feeds.csv not found in {FEEDS_CANDIDATES}")

feeds_df = pd.read_csv(FEEDS_CSV)
feed_to_section = dict(zip(feeds_df["feed_name"], feeds_df["section"]))

# 최근 N시간(기본 24h) 데이터만
HOURS = 24
since = (datetime.now(timezone.utc) - timedelta(hours=HOURS)).isoformat()

conn = sqlite3.connect(DB_PATH)
q = """
SELECT item_id, feed_name, title, summary, link, published, fetched_at
FROM items
WHERE fetched_at >= ?
"""
df = pd.read_sql_query(q, conn, params=[since])
df["section"] = df["feed_name"].map(feed_to_section)

print("loaded rows:", len(df), "| since:", since)
df.head(3)

loaded rows: 557 | since: 2026-05-13T09:06:18.596898+00:00


,item_id,feed_name,title,summary,link,published,fetched_at,section
0,991158f5c2c4353b7260c8a238f0e3f923da8fab38c98b...,donga_politics,"李대통령 “농협, 불투명한 의사결정 구조 등 개선에 속도내야”","<img align=""left"" hspace=""10"" src=""https://dim...",https://www.donga.com/news/Politics/article/al...,"Thu, 14 May 2026 17:55:47 +0900",2026-05-14T09:06:16.863336+00:00,politics
1,a6e558d861dc437ca5e31737f98e7963fa81407b3c4502...,donga_politics,하정우 “북구 1인당 지역총생산 1억2000만원” 발언에…한동훈-박민식 비판,"<img align=""left"" hspace=""10"" src=""https://dim...",https://www.donga.com/news/Politics/article/al...,"Thu, 14 May 2026 17:48:00 +0900",2026-05-14T09:06:16.863336+00:00,politics
2,ab1dd7031fbf819a4a9862df960de08d3708f2297f6141...,donga_politics,조국당·민주당 김상욱으로 울산시장 후보 단일화…진보당과는 협의 중,"<img align=""left"" hspace=""10"" src=""https://dim...",https://www.donga.com/news/Politics/article/al...,"Thu, 14 May 2026 17:42:00 +0900",2026-05-14T09:06:16.863336+00:00,politics


In [12]:
import re
from html import unescape

TAG_RE = re.compile(r"<[^>]+>")

def clean_text(x):
    if not isinstance(x, str):
        return ""
    x = unescape(x)
    x = TAG_RE.sub(" ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

df["title_clean"] = df["title"].map(clean_text)
df["summary_clean"] = df["summary"].map(clean_text)
df["text"] = (df["title_clean"] + " " + df["summary_clean"]).str.strip()

# 너무 짧은 건 제거(클러스터링 노이즈 방지)
df2 = df[df["text"].str.len() >= 15].copy()
print("usable rows:", len(df2))
df2[["section","feed_name","title_clean"]].head(5)

usable rows: 557


,section,feed_name,title_clean
0,politics,donga_politics,"李대통령 “농협, 불투명한 의사결정 구조 등 개선에 속도내야”"
1,politics,donga_politics,하정우 “북구 1인당 지역총생산 1억2000만원” 발언에…한동훈-박민식 비판
2,politics,donga_politics,조국당·민주당 김상욱으로 울산시장 후보 단일화…진보당과는 협의 중
3,politics,donga_politics,[정치 한 컷]오랜만에 보는 ‘여야 웃음’
4,politics,donga_politics,中 “트럼프와 한반도 문제도 논의”…김정은 메시지 전달했나?


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN
import numpy as np

# 정치만 먼저 테스트
sec = "politics"
sub = df2[df2["section"] == sec].copy()
print("section:", sec, "rows:", len(sub))

# 한국어는 형태소 분석 없이 char n-gram이 안정적
vec = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2
)
X = vec.fit_transform(sub["text"])

# eps는 데이터에 따라 튜닝 포인트
clu = DBSCAN(eps=0.55, min_samples=2, metric="cosine")
labels = clu.fit_predict(X)
sub["cluster"] = labels

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = (labels == -1).sum()

print("clusters:", n_clusters, "| noise:", n_noise)
sub["cluster"].value_counts().head(10)

section: politics rows: 144
clusters: 30 | noise: 78


cluster
-1     78
 5      4
 0      3
 6      3
 17     3
 19     3
 1      2
 2      2
 3      2
 4      2
Name: count, dtype: int64

In [14]:
from collections import defaultdict

def 대표기사_인덱스(Xc):
    # centroid와 가장 유사한(=dot이 큰) 문서를 대표로
    centroid = Xc.mean(axis=0)
    sims = (Xc @ centroid.T).A.ravel()
    return int(np.argmax(sims))

rows = []
for c, g in sub[sub["cluster"] != -1].groupby("cluster"):
    idxs = g.index.to_list()
    Xc = X[[sub.index.get_loc(i) for i in idxs], :]
    rep_pos = 대표기사_인덱스(Xc)
    rep_idx = idxs[rep_pos]

    top_titles = g.loc[idxs, "title_clean"].head(3).to_list()
    rows.append({
        "cluster": int(c),
        "size": len(g),
        "representative_title": sub.loc[rep_idx, "title_clean"],
        "top3_titles": " / ".join(top_titles)
    })

summary = pd.DataFrame(rows).sort_values(["size","cluster"], ascending=[False, True])
summary.head(10)

,cluster,size,representative_title,top3_titles
5,5,4,"李, 새마을운동중앙회 찾아 “박정희때 큰 성과…지금도 유용”","새마을운동중앙회 찾은 李 “봉사활동 가장 잘하는 단체일 것” / 李, 새마을운동중앙..."
0,0,3,"李대통령 “농협, 불투명한 의사결정 구조 등 개선에 속도내야”","李대통령 “농협, 불투명한 의사결정 구조 등 개선에 속도내야” / 이 대통령 “농협..."
6,6,3,"고위당국자 “이란 외 주체, 나무호 공격 가능성 낮아”","외교부 고위급 “나무호 공격 주체, 이란 아닐 가능성 낮아” / 고위당국자 “이란 ..."
17,17,3,22대 국회 후반기 국회의장 후보에 ‘대통령 정무특보’ 출신 조정식 선출,"與, 후반기 국회의장 후보 ‘친명’ 조정식 선출 / 후반기 국회의장에 조정식 사실상..."
19,19,3,김용범이 쏘아올린 AI 시대 국민배당금 논쟁…야당의 ‘기업 이윤 강탈’ 비틀기 속 ...,국민배당금 논쟁에 여당 내에서도 “지극히 옳은 말” “정제된 발언 해야” / [뉴스...
1,1,2,조국당·민주당 김상욱으로 울산시장 후보 단일화…진보당과는 협의 중,"조국당·민주당 김상욱으로 울산시장 후보 단일화…진보당과는 협의 중 / ""울산시장 민..."
2,2,2,[정치 한 컷]오랜만에 보는 ‘여야 웃음’,[정치 한 컷]오랜만에 보는 ‘여야 웃음’ / 여야 원내대표 상견례…20일 '의장 ...
3,3,2,中 “트럼프와 한반도 문제도 논의”…김정은 메시지 전달했나?,"中 “트럼프와 한반도 문제도 논의”…김정은 메시지 전달했나? / 미·중 정상, 한반..."
4,4,2,"“요격 성공률 96%인 천궁-2, 전원 공급 없어도 10분내 발사”","“요격 성공률 96%인 천궁-2, 전원 공급 없어도 10분내 발사” / 북 탄도미사..."
7,7,2,"건강 악화된 정이한 후보, 박형준 설득 끝 단식 농성 종료","건강 악화된 정이한 후보, 박형준 설득 끝 단식 농성 종료 / [6·3부산]전재수·..."


In [15]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# 1) feeds.csv 위치 찾기 (너는 data/configs 쪽에 있음)
ROOT = Path.cwd()
for p in [ROOT] + list(ROOT.parents):
    if (p / "db").exists() and (p / "notebooks").exists():
        PROJECT_ROOT = p
        break
else:
    PROJECT_ROOT = ROOT

candidates = [
    PROJECT_ROOT / "configs" / "feeds.csv",
    PROJECT_ROOT / "data" / "configs" / "feeds.csv",
]
FEEDS_CSV = next((p for p in candidates if p.exists()), None)
if FEEDS_CSV is None:
    raise FileNotFoundError(f"feeds.csv not found. tried: {candidates}")

feeds_df = pd.read_csv(FEEDS_CSV)

# 2) politics_bucket 컬럼 보장 + 문자열 dtype으로 강제
if "politics_bucket" not in feeds_df.columns:
    feeds_df["politics_bucket"] = ""

feeds_df["politics_bucket"] = feeds_df["politics_bucket"].fillna("").astype("string")

# 3) 기존 정치 피드 bucket 패치
bucket_patch = {
    "donga_politics": "conservative",
    "khan_politics": "progressive",
}
mask = feeds_df["feed_name"].isin(bucket_patch.keys())
feeds_df.loc[mask, "politics_bucket"] = (
    feeds_df.loc[mask, "feed_name"].map(bucket_patch).astype("string")
)

# 4) 추가할 피드들 (원하면 이후에 더 늘리면 됨)
new_rows = [
    # 한겨레
    {"publisher":"hani", "section":"politics",      "feed_name":"hani_politics",      "feed_url":"https://www.hani.co.kr/rss/politics/",       "politics_bucket":"progressive"},
    {"publisher":"hani", "section":"economy",       "feed_name":"hani_economy",       "feed_url":"https://www.hani.co.kr/rss/economy/",        "politics_bucket":""},
    {"publisher":"hani", "section":"society",       "feed_name":"hani_society",       "feed_url":"https://www.hani.co.kr/rss/society/",        "politics_bucket":""},
    {"publisher":"hani", "section":"international", "feed_name":"hani_international", "feed_url":"https://www.hani.co.kr/rss/international/",  "politics_bucket":""},

    # 연합뉴스TV
    {"publisher":"yonhapnewstv", "section":"politics",      "feed_name":"yntv_politics",      "feed_url":"http://www.yonhapnewstv.co.kr/category/news/politics/feed/",       "politics_bucket":"centrist"},
    {"publisher":"yonhapnewstv", "section":"economy",       "feed_name":"yntv_economy",       "feed_url":"http://www.yonhapnewstv.co.kr/category/news/economy/feed/",        "politics_bucket":""},
    {"publisher":"yonhapnewstv", "section":"society",       "feed_name":"yntv_society",       "feed_url":"http://www.yonhapnewstv.co.kr/category/news/society/feed/",        "politics_bucket":""},
    {"publisher":"yonhapnewstv", "section":"international", "feed_name":"yntv_international", "feed_url":"http://www.yonhapnewstv.co.kr/category/news/international/feed/",  "politics_bucket":""},
]
add_df = pd.DataFrame(new_rows)
add_df["politics_bucket"] = add_df["politics_bucket"].fillna("").astype("string")

# 5) 중복 feed_name 제거 후 병합
existing = set(feeds_df["feed_name"].astype(str))
add_df = add_df[~add_df["feed_name"].astype(str).isin(existing)].copy()

feeds_df2 = pd.concat([feeds_df, add_df], ignore_index=True)

# 6) 저장
feeds_df2.to_csv(FEEDS_CSV, index=False, encoding="utf-8-sig")
print("updated:", FEEDS_CSV)
print("total rows:", len(feeds_df2))

# 7) 정치 피드 3관점 확인
politics_view = feeds_df2[feeds_df2["section"].astype(str) == "politics"][["publisher","feed_name","politics_bucket","feed_url"]]
display(politics_view)

updated: /home/epistachio/workspace/news-briefing/configs/feeds.csv
total rows: 20


,publisher,feed_name,politics_bucket,feed_url
0,donga,donga_politics,conservative,https://rss.donga.com/politics.xml
4,khan,khan_politics,progressive,https://www.khan.co.kr/rss/rssdata/politic_new...
8,newsis,newsis_politics,centrist,https://www.newsis.com/RSS/politics.xml
12,hani,hani_politics,progressive,https://www.hani.co.kr/rss/politics/
16,yonhapnewstv,yntv_politics,centrist,http://www.yonhapnewstv.co.kr/category/news/po...


In [16]:
from pathlib import Path
import sqlite3
import pandas as pd
import feedparser
from datetime import datetime, timezone
import hashlib
from urllib.parse import urlparse, urlunparse
from tqdm import tqdm

# ---------- project root ----------
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "src").exists() and (p / "notebooks").exists() and (p / "db").exists():
            return p
    return start

ROOT = find_project_root(Path.cwd())

# ---------- feeds.csv 탐색 (네 케이스: data/configs/feeds.csv) ----------
FEEDS_CANDIDATES = [
    ROOT / "configs" / "feeds.csv",
    ROOT / "data" / "configs" / "feeds.csv",
]
FEEDS_CSV = next((p for p in FEEDS_CANDIDATES if p.exists()), None)
if FEEDS_CSV is None:
    raise FileNotFoundError(f"feeds.csv not found. tried: {FEEDS_CANDIDATES}")

feeds_df = pd.read_csv(FEEDS_CSV)
print("feeds.csv:", FEEDS_CSV, "| rows:", len(feeds_df))

# ---------- URL 정규화 + item_id ----------
def normalize_url(u: str | None) -> str | None:
    if not isinstance(u, str) or not u.strip():
        return None
    p = urlparse(u.strip())
    return urlunparse((p.scheme, p.netloc, p.path, "", "", ""))  # query/fragment 제거

def make_item_id(feed_url: str | None, guid: str | None, link: str | None, title: str | None, published: str | None) -> str:
    base = guid or normalize_url(link) or f"{title}|{published}|{feed_url}"
    base = (base or "").strip()
    return hashlib.sha256(base.encode("utf-8")).hexdigest()

def fetch_feed(feed_name: str, feed_url: str, limit: int = 50) -> pd.DataFrame:
    d = feedparser.parse(feed_url)
    fetched_at = datetime.now(timezone.utc).isoformat()

    rows = []
    for e in d.entries[:limit]:
        link = getattr(e, "link", None)
        guid = getattr(e, "id", None) or getattr(e, "guid", None)
        title = getattr(e, "title", None)
        published = getattr(e, "published", None)
        summary = getattr(e, "summary", None)

        link_norm = normalize_url(link)
        item_id = make_item_id(feed_url, guid, link_norm, title, published)

        rows.append({
            "item_id": item_id,
            "feed_name": feed_name,
            "feed_url": feed_url,
            "title": title,
            "link": link,
            "link_norm": link_norm,
            "published": published,
            "summary": summary,
            "guid": guid,
            "fetched_at": fetched_at,
        })
    return pd.DataFrame(rows)

# ---------- SQLite ----------
DB_PATH = ROOT / "db" / "news.db"
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS items (
    item_id TEXT PRIMARY KEY,
    feed_name TEXT,
    feed_url TEXT,
    title TEXT,
    link TEXT,
    link_norm TEXT,
    published TEXT,
    summary TEXT,
    guid TEXT,
    fetched_at TEXT
)
""")
conn.commit()

before = cur.execute("SELECT COUNT(*) FROM items").fetchone()[0]

report = []
for _, r in tqdm(feeds_df.iterrows(), total=len(feeds_df)):
    fn = str(r["feed_name"])
    url = str(r["feed_url"])

    try:
        df = fetch_feed(fn, url, limit=50)
        fetched = len(df)

        if fetched == 0:
            report.append({"feed_name": fn, "fetched": 0, "inserted": 0, "status": "empty"})
            continue

        records = df[[
            "item_id","feed_name","feed_url","title","link","link_norm","published","summary","guid","fetched_at"
        ]].to_records(index=False)

        cur.executemany("""
        INSERT OR IGNORE INTO items
        (item_id, feed_name, feed_url, title, link, link_norm, published, summary, guid, fetched_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, list(records))
        conn.commit()

        # inserted 계산: 현재 총량 - (before + 지금까지 inserted 누적)
        current_total = cur.execute("SELECT COUNT(*) FROM items").fetchone()[0]
        inserted_so_far = sum(x["inserted"] for x in report)
        inserted = current_total - (before + inserted_so_far)

        report.append({"feed_name": fn, "fetched": fetched, "inserted": inserted, "status": "ok"})

    except Exception as e:
        report.append({"feed_name": fn, "fetched": 0, "inserted": 0, "status": f"error: {type(e).__name__}"})

after = cur.execute("SELECT COUNT(*) FROM items").fetchone()[0]
rep_df = pd.DataFrame(report).sort_values(["status","inserted","fetched"], ascending=[True, False, False])

print("DB:", DB_PATH)
print("items before:", before, "after:", after, "delta:", after - before)

# 새로 추가한 정치 피드들만 눈에 띄게 확인
rep_df[rep_df["feed_name"].isin(["hani_politics","yntv_politics","hani_economy","yntv_economy","hani_society","yntv_society","hani_international","yntv_international"])]

feeds.csv: /home/epistachio/workspace/news-briefing/configs/feeds.csv | rows: 20


100%|██████████████████████████████████████████████████████| 20/20 [00:03<00:00,  5.91it/s]

DB: /home/epistachio/workspace/news-briefing/db/news.db
items before: 1960 after: 2114 delta: 154


,feed_name,fetched,inserted,status
12,hani_politics,30,30,ok
13,hani_economy,30,30,ok
14,hani_society,29,29,ok
15,hani_international,30,21,ok
16,yntv_politics,11,11,ok
17,yntv_economy,11,11,ok
18,yntv_society,11,11,ok
19,yntv_international,11,11,ok


In [17]:
from pathlib import Path
import sqlite3
import pandas as pd
from datetime import datetime, timedelta, timezone
import re
from html import unescape

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "src").exists() and (p / "notebooks").exists() and (p / "db").exists():
            return p
    return start

ROOT = find_project_root(Path.cwd())
DB_PATH = ROOT / "db" / "news.db"

# feeds.csv 위치 (너 케이스)
FEEDS_CSV = next((p for p in [ROOT/"configs/feeds.csv", ROOT/"data/configs/feeds.csv"] if p.exists()), None)
if FEEDS_CSV is None:
    raise FileNotFoundError("feeds.csv not found")

feeds_df = pd.read_csv(FEEDS_CSV)
feed_to_section = dict(zip(feeds_df["feed_name"], feeds_df["section"]))
feed_to_bucket  = dict(zip(feeds_df["feed_name"], feeds_df["politics_bucket"].fillna("")))

HOURS = 24
since = (datetime.now(timezone.utc) - timedelta(hours=HOURS)).isoformat()

conn = sqlite3.connect(DB_PATH)
q = """
SELECT item_id, feed_name, title, summary, link, published, fetched_at
FROM items
WHERE fetched_at >= ?
"""
df = pd.read_sql_query(q, conn, params=[since])

df["section"] = df["feed_name"].map(feed_to_section)
df["politics_bucket"] = df["feed_name"].map(feed_to_bucket).fillna("")

# 정치만
pol = df[df["section"] == "politics"].copy()

# 텍스트 클린
TAG_RE = re.compile(r"<[^>]+>")
def clean_text(x):
    if not isinstance(x, str):
        return ""
    x = unescape(x)
    x = TAG_RE.sub(" ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

pol["title_clean"] = pol["title"].map(clean_text)
pol["summary_clean"] = pol["summary"].map(clean_text)
pol["text"] = (pol["title_clean"] + " " + pol["summary_clean"]).str.strip()

pol = pol[pol["text"].str.len() >= 15].copy()

print("loaded politics rows:", len(pol), "| buckets:", pol["politics_bucket"].value_counts(dropna=False).to_dict())
pol[["feed_name","politics_bucket","title_clean"]].head(5)

loaded politics rows: 185 | buckets: {'progressive': 74, 'centrist': 61, 'conservative': 50}


,feed_name,politics_bucket,title_clean
0,donga_politics,conservative,"李대통령 “농협, 불투명한 의사결정 구조 등 개선에 속도내야”"
1,donga_politics,conservative,하정우 “북구 1인당 지역총생산 1억2000만원” 발언에…한동훈-박민식 비판
2,donga_politics,conservative,조국당·민주당 김상욱으로 울산시장 후보 단일화…진보당과는 협의 중
3,donga_politics,conservative,[정치 한 컷]오랜만에 보는 ‘여야 웃음’
4,donga_politics,conservative,中 “트럼프와 한반도 문제도 논의”…김정은 메시지 전달했나?


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN
import numpy as np

vec = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2
)
X = vec.fit_transform(pol["text"])

# eps는 튜닝 포인트 (일단 0.55로 시작)
clu = DBSCAN(eps=0.55, min_samples=2, metric="cosine")
labels = clu.fit_predict(X)
pol["cluster"] = labels

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = int((labels == -1).sum())

print("clusters:", n_clusters, "| noise:", n_noise)
pol["cluster"].value_counts().head(10)

clusters: 35 | noise: 107


cluster
-1     107
 5       5
 0       3
 6       3
 19      3
 20      3
 21      3
 1       2
 2       2
 3       2
Name: count, dtype: int64

In [19]:
def representative_by_bucket(pol_df: pd.DataFrame, X):
    # 클러스터 단위로 centroid를 만들고, bucket별로 centroid와 가장 유사한 문서를 대표로 선택
    rows = []
    buckets = ["conservative", "progressive", "centrist"]

    # 인덱스 매핑(행 위치 -> sparse row)
    idx_to_pos = {idx: pos for pos, idx in enumerate(pol_df.index)}

    for c, g in pol_df[pol_df["cluster"] != -1].groupby("cluster"):
        idxs = g.index.tolist()
        Xc = X[[idx_to_pos[i] for i in idxs], :]
        centroid = Xc.mean(axis=0)

        row = {
            "cluster": int(c),
            "size": int(len(g)),
            "sources": int(g["feed_name"].nunique()),
        }

        for b in buckets:
            gb = g[g["politics_bucket"] == b]
            if len(gb) == 0:
                row[f"{b}_title"] = ""
                row[f"{b}_link"] = ""
                continue

            bidxs = gb.index.tolist()
            Xb = X[[idx_to_pos[i] for i in bidxs], :]
            sims = (Xb @ centroid.T).A.ravel()
            rep = bidxs[int(np.argmax(sims))]

            row[f"{b}_title"] = pol_df.loc[rep, "title_clean"]
            row[f"{b}_link"] = pol_df.loc[rep, "link"]

        rows.append(row)

    out = pd.DataFrame(rows).sort_values(["size","cluster"], ascending=[False, True])
    return out

cmp = representative_by_bucket(pol, X)
cmp.head(10)

,cluster,size,sources,conservative_title,conservative_link,progressive_title,progressive_link,centrist_title,centrist_link
5,5,5,4,"李, 새마을운동중앙회 찾아 “박정희때 큰 성과…지금도 유용”",https://www.donga.com/news/Politics/article/al...,"이 대통령 “박정희 대통령의 성과, 지금도 유용”···민주당 출신 현직 최초로 ‘새...",https://www.khan.co.kr/article/202605141647001...,"이 대통령 ""새마을운동, 박정희 시작해 큰 성과…지금도 유용""",https://www.yonhapnewstv.co.kr/news/AKR2026051...
0,0,3,3,"李대통령 “농협, 불투명한 의사결정 구조 등 개선에 속도내야”",https://www.donga.com/news/Politics/article/al...,이 대통령 “농협 정상화 무엇보다 중요…구조적 병폐 바로잡아야”,https://www.khan.co.kr/article/202605141742001...,"이 대통령 ""농어촌 기본소득·햇빛소득 확대…농협 정상화 중요""",https://www.newsis.com/view/NISX20260514_00036...
6,6,3,2,"고위당국자 “이란 외 주체, 나무호 공격 가능성 낮아”",https://www.donga.com/news/Inter/article/all/2...,"고위 당국자 “이란 외 공격 가능성 상식적으로 크지 않아”…정부, 기술분석팀 두바이 파견",https://www.khan.co.kr/article/202605141630001...,,
19,19,3,2,"與, 후반기 국회의장 후보 ‘친명’ 조정식 선출",https://www.donga.com/news/Politics/article/al...,22대 국회 후반기 국회의장 후보에 ‘대통령 정무특보’ 출신 조정식 선출,https://www.khan.co.kr/article/202605131820001...,,
20,20,3,2,,,[선택! 6·3지방선거]박찬대·유정복 인천시장 후보 등록···‘선거전’ 돌입,https://www.khan.co.kr/article/202605141525001...,"박찬대·유정복, 후보 등록…""결코 질수없다"" 선거전 돌입[6·3인천]",https://www.newsis.com/view/NISX20260514_00036...
21,21,3,1,,,김용범이 쏘아올린 AI 시대 국민배당금 논쟁…야당의 ‘기업 이윤 강탈’ 비틀기 속 ...,https://www.khan.co.kr/article/202605131703001...,,
1,1,2,2,조국당·민주당 김상욱으로 울산시장 후보 단일화…진보당과는 협의 중,https://www.donga.com/news/Politics/article/al...,,,"""울산시장 민주 김상욱 후보로 단일화"" 혁신 황명필의 선택",https://www.newsis.com/view/NISX20260514_00036...
2,2,2,2,[정치 한 컷]오랜만에 보는 ‘여야 웃음’,https://www.donga.com/news/Politics/article/al...,,,여야 원내대표 상견례…20일 '의장 선출' 본회의 일정 합의 불발,https://www.newsis.com/view/NISX20260514_00036...
3,3,2,2,中 “트럼프와 한반도 문제도 논의”…김정은 메시지 전달했나?,https://www.donga.com/news/Politics/article/al...,"미·중 정상, 한반도 문제 의견 교환…전문가들 “우선순위 밀려 원론적 논의 가능성”",https://www.khan.co.kr/article/202605141757001...,,
4,4,2,2,"“요격 성공률 96%인 천궁-2, 전원 공급 없어도 10분내 발사”",https://www.donga.com/news/Politics/article/al...,북 탄도미사일 쏘면 10여분 내 “요격 준비 끝”…중동서 위력 떨친 ‘천궁-Ⅱ’ 이...,https://www.khan.co.kr/article/202605141532001...,,


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN
import numpy as np
import pandas as pd

# 벡터는 고정(클러스터링 파라미터만 바꾸기)
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2)
X = vec.fit_transform(pol["text"])

buckets = ["conservative", "progressive", "centrist"]

def bucket_coverage_stats(df, labels):
    df = df.copy()
    df["cluster"] = labels

    # noise 제외
    core = df[df["cluster"] != -1]
    if len(core) == 0:
        return dict(n_clusters=0, noise=len(df), noise_rate=1.0, c3=0, c2=0, avg_size=0)

    grp = core.groupby("cluster")

    n_clusters = grp.ngroups
    noise = int((labels == -1).sum())
    noise_rate = noise / len(df)

    # 클러스터별 bucket 커버 수
    cover = grp["politics_bucket"].apply(lambda s: len(set([x for x in s if x in buckets])))
    c3 = int((cover >= 3).sum())
    c2 = int((cover >= 2).sum())
    avg_size = float(grp.size().mean())

    return dict(n_clusters=n_clusters, noise=noise, noise_rate=noise_rate, c3=c3, c2=c2, avg_size=avg_size)

eps_list = [0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

rows = []
for eps in eps_list:
    labels = DBSCAN(eps=eps, min_samples=2, metric="cosine").fit_predict(X)
    stats = bucket_coverage_stats(pol, labels)
    rows.append({"eps": eps, **stats})

sweep = pd.DataFrame(rows).sort_values(
    ["c3", "c2", "noise_rate", "n_clusters", "avg_size"],
    ascending=[False, False, True, False, False]
).reset_index(drop=True)

display(sweep)

best_eps = float(sweep.loc[0, "eps"])
print("BEST eps =", best_eps)

,eps,n_clusters,noise,noise_rate,c3,c2,avg_size
0,0.70,38,66,0.356757,2,19,3.131579
1,0.60,33,98,0.529730,2,17,2.636364
2,0.55,35,107,0.578378,2,17,2.228571
3,0.65,34,83,0.448649,2,16,3.000000
4,0.50,29,124,0.670270,1,11,2.103448
5,0.45,23,137,0.740541,1,7,2.086957


BEST eps = 0.7


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN
import numpy as np
import pandas as pd

EPS = 0.70
buckets = ["conservative", "progressive", "centrist"]

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2)
X = vec.fit_transform(pol["text"])

labels = DBSCAN(eps=EPS, min_samples=2, metric="cosine").fit_predict(X)
pol2 = pol.copy()
pol2["cluster"] = labels

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = int((labels == -1).sum())
print("EPS =", EPS, "| clusters:", n_clusters, "| noise:", n_noise)
pol2["cluster"].value_counts().head(10)

EPS = 0.7 | clusters: 38 | noise: 66


cluster
-1     66
 7     18
 0     11
 11     6
 21     6
 6      5
 8      3
 10     3
 17     3
 22     3
Name: count, dtype: int64

In [22]:
def representative_by_bucket(pol_df: pd.DataFrame, X):
    idx_to_pos = {idx: pos for pos, idx in enumerate(pol_df.index)}
    rows = []

    for c, g in pol_df[pol_df["cluster"] != -1].groupby("cluster"):
        idxs = g.index.tolist()
        Xc = X[[idx_to_pos[i] for i in idxs], :]
        centroid = Xc.mean(axis=0)

        cover = [b for b in buckets if (g["politics_bucket"] == b).any()]
        row = {
            "cluster": int(c),
            "size": int(len(g)),
            "sources": int(g["feed_name"].nunique()),
            "bucket_cover": len(cover),
        }

        for b in buckets:
            gb = g[g["politics_bucket"] == b]
            if len(gb) == 0:
                row[f"{b}_title"] = ""
                row[f"{b}_link"] = ""
                continue

            bidxs = gb.index.tolist()
            Xb = X[[idx_to_pos[i] for i in bidxs], :]
            sims = (Xb @ centroid.T).A.ravel()
            rep = bidxs[int(np.argmax(sims))]

            row[f"{b}_title"] = pol_df.loc[rep, "title_clean"]
            row[f"{b}_link"] = pol_df.loc[rep, "link"]

        rows.append(row)

    out = pd.DataFrame(rows).sort_values(
        ["bucket_cover", "size", "sources", "cluster"],
        ascending=[False, False, False, True]
    ).reset_index(drop=True)
    return out

cmp = representative_by_bucket(pol2, X)
cmp.head(15)

,cluster,size,sources,bucket_cover,conservative_title,conservative_link,progressive_title,progressive_link,centrist_title,centrist_link
0,7,18,3,3,6·3 지선·재보선 오늘부터 후보등록…정원오·오세훈 첫날 등록,https://www.donga.com/news/Politics/article/al...,[선택! 6·3지방선거]박찬대·유정복 인천시장 후보 등록···‘선거전’ 돌입,https://www.khan.co.kr/article/202605141525001...,[6·3경기북부]시장·군수 후보 등록 잇따라…본격 선거전(종합),https://www.newsis.com/view/NISX20260514_00036...
1,0,11,5,3,"李, 새마을운동중앙회 찾아 “박정희때 큰 성과…지금도 유용”",https://www.donga.com/news/Politics/article/al...,"이 대통령 “박정희 대통령의 성과, 지금도 유용”···민주당 출신 현직 최초로 ‘새...",https://www.khan.co.kr/article/202605141647001...,"이 대통령 ""새마을운동, 박정희 시작해 큰 성과…지금도 유용""",https://www.yonhapnewstv.co.kr/news/AKR2026051...
2,11,6,2,2,"李 “김용범, AI 초과세수 국민배당 방안 말한것”",https://www.donga.com/news/Politics/article/al...,김용범이 쏘아올린 AI 시대 국민배당금 논쟁…야당의 ‘기업 이윤 강탈’ 비틀기 속 ...,https://www.khan.co.kr/article/202605131703001...,,
3,21,6,2,2,"與, 후반기 국회의장 후보 ‘친명’ 조정식 선출",https://www.donga.com/news/Politics/article/al...,22대 국회 후반기 국회의장 후보에 ‘대통령 정무특보’ 출신 조정식 선출,https://www.khan.co.kr/article/202605131820001...,,
4,6,5,3,2,"고위당국자 “이란 외 주체, 나무호 공격 가능성 낮아”",https://www.donga.com/news/Inter/article/all/2...,고위 당국자 “이란 외 주체가 나무호 공격 가능성 낮아”,https://www.hani.co.kr/arti/politics/politics_...,,
5,30,3,3,2,,,"‘한 자릿수’ 지지율 차에…정원오 “은퇴 세대 1주택자, 재산세 감면”",https://www.khan.co.kr/article/202605132053005...,"김병욱 ""1세대 1주택 취약계층, 재산세 한시 감면 추진""[6·3성남]",https://www.newsis.com/view/NISX20260514_00036...
6,8,3,2,2,"정원오측, “鄭, 폭행사건 수습하려다 휘말렸다” 동석자 주장 공개",https://www.donga.com/news/Politics/article/al...,,,"정원오 측, 폭행사건 동석자 입장 공개…""鄭, 상황 수습하다 휘말려""",https://www.newsis.com/view/NISX20260514_00036...
7,17,3,2,2,"안철수, 박형준 부산선대위 합류…공동 명예선대위원장",https://www.donga.com/news/Politics/article/al...,"박형준 부산시장 후보, 명예 선대위원장에 안철수 위촉…“보수 대통합” 기대",https://www.khan.co.kr/article/202605140835001...,,
8,22,3,2,2,"李, 베선트 만나 한미 통화스와프 체결 요청",https://www.donga.com/news/Politics/article/al...,"이 대통령, 미·중 정상회담 앞 베선트·허리펑 잇따라 만나…협력 의지 재확인",https://www.khan.co.kr/article/202605131641001...,,
9,2,2,2,2,조국당·민주당 김상욱으로 울산시장 후보 단일화…진보당과는 협의 중,https://www.donga.com/news/Politics/article/al...,,,"""울산시장 민주 김상욱 후보로 단일화"" 혁신 황명필의 선택",https://www.newsis.com/view/NISX20260514_00036...


In [23]:
from pathlib import Path
from datetime import datetime

bucket_kr = {"conservative":"보수", "progressive":"진보", "centrist":"중도"}

def format_issue(row, i):
    lines = []
    lines.append(f"### 정치 이슈 {i} (기사 {row['size']} / 매체 {row['sources']} / 관점 {row['bucket_cover']}종)")
    for b in buckets:
        title = row.get(f"{b}_title", "") or ""
        link  = row.get(f"{b}_link", "") or ""
        if title.strip():
            lines.append(f"- [{bucket_kr[b]}] {title} — {link}")
        else:
            lines.append(f"- [{bucket_kr[b]}] (해당 관점 기사 없음)")
    return "\n".join(lines)

TOP_N = 10
selected = cmp.head(TOP_N).copy()

brief = []
brief.append(f"# 아침 정치 브리핑 (eps={EPS})")
brief.append(f"- 생성 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
brief.append("")
for i, (_, row) in enumerate(selected.iterrows(), start=1):
    brief.append(format_issue(row, i))
    brief.append("")

brief_text = "\n".join(brief)
print(brief_text[:1200])  # 미리보기(앞부분만)

OUT_PATH = Path("../data/processed/politics_briefing.md")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUT_PATH.write_text(brief_text, encoding="utf-8")
print("\nsaved:", OUT_PATH.resolve())

# 아침 정치 브리핑 (eps=0.7)
- 생성 시각: 2026-05-14 18:06:23

### 정치 이슈 1 (기사 18 / 매체 3 / 관점 3종)
- [보수] 6·3 지선·재보선 오늘부터 후보등록…정원오·오세훈 첫날 등록 — https://www.donga.com/news/Politics/article/all/20260514/133919339/1
- [진보] [선택! 6·3지방선거]박찬대·유정복 인천시장 후보 등록···‘선거전’ 돌입 — https://www.khan.co.kr/article/202605141525001/?utm_source=khan_rss&utm_medium=rss&utm_campaign=politic_news
- [중도] [6·3경기북부]시장·군수 후보 등록 잇따라…본격 선거전(종합) — https://www.newsis.com/view/NISX20260514_0003630156

### 정치 이슈 2 (기사 11 / 매체 5 / 관점 3종)
- [보수] 李, 새마을운동중앙회 찾아 “박정희때 큰 성과…지금도 유용” — https://www.donga.com/news/Politics/article/all/20260514/133924494/1
- [진보] 이 대통령 “박정희 대통령의 성과, 지금도 유용”···민주당 출신 현직 최초로 ‘새마을운동회’ 방문 — https://www.khan.co.kr/article/202605141647001/?utm_source=khan_rss&utm_medium=rss&utm_campaign=politic_news
- [중도] 이 대통령 "새마을운동, 박정희 시작해 큰 성과…지금도 유용" — https://www.yonhapnewstv.co.kr/news/AKR20260514155308SX7

### 정치 이슈 3 (기사 6 / 매체 2 / 관점 2종)
- [보수] 李 “김용범, AI 초과세수 국민배당 방안 말한것” — https://www.donga.com/news/Politics/article/al

In [24]:
from pathlib import Path

# 1) 현재 위치 확인
print("CWD =", Path.cwd())

# 2) 루트 탐색: db/news.db가 있는 상위 폴더를 루트로
def find_root_by_db(start: Path) -> Path:
    for p in [start.resolve()] + list(start.resolve().parents):
        if (p / "db" / "news.db").exists():
            return p
    raise FileNotFoundError("Cannot find project root containing db/news.db")

ROOT = find_root_by_db(Path.cwd())
print("ROOT =", ROOT)

# 3) 저장 경로를 루트 기준으로 고정
OUT_PATH = ROOT / "data" / "processed" / "politics_briefing.md"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# 4) 방금 만든 brief_text가 노트북에 남아있다면 그대로 다시 저장
# (없으면 NameError 뜰 텐데, 그럼 brief_text 생성 셀을 한 번만 다시 돌리면 됨)
OUT_PATH.write_text(brief_text, encoding="utf-8")
print("saved:", OUT_PATH)

CWD = /home/epistachio/workspace/news-briefing/data/raw
ROOT = /home/epistachio/workspace/news-briefing/data
saved: /home/epistachio/workspace/news-briefing/data/data/processed/politics_briefing.md


In [25]:
from pathlib import Path
import os, shutil

CANONICAL_ROOT = Path.home() / "workspace" / "news-briefing"
assert CANONICAL_ROOT.exists(), f"not found: {CANONICAL_ROOT}"

# 1) 노트북 작업 디렉토리를 프로젝트 루트로 강제 이동
os.chdir(CANONICAL_ROOT)
print("CWD =", Path.cwd())

# 2) feeds.csv를 '정상 위치(root/configs)'로 복사(없으면 만들기)
src_feeds = CANONICAL_ROOT / "data" / "configs" / "feeds.csv"   # 지금까지 너가 만든 위치
dst_feeds = CANONICAL_ROOT / "configs" / "feeds.csv"           # 앞으로 쓸 표준 위치

dst_feeds.parent.mkdir(parents=True, exist_ok=True)

if src_feeds.exists():
    shutil.copy2(src_feeds, dst_feeds)
    print("feeds.csv copied ->", dst_feeds)
else:
    print("feeds.csv source missing:", src_feeds)

# 3) 브리핑 저장 경로를 '정상 위치(root/data/processed)'로 고정해서 다시 저장
dst_out = CANONICAL_ROOT / "data" / "processed" / "politics_briefing.md"
dst_out.parent.mkdir(parents=True, exist_ok=True)

# (a) notebook에 brief_text가 있으면 그걸로 저장
try:
    dst_out.write_text(brief_text, encoding="utf-8")
    print("briefing saved(from brief_text) ->", dst_out)
except NameError:
    # (b) 없으면, 잘못 저장된 파일에서 복사
    bad_out = CANONICAL_ROOT / "data" / "data" / "processed" / "politics_briefing.md"
    if bad_out.exists():
        shutil.copy2(bad_out, dst_out)
        print("briefing copied ->", dst_out)
    else:
        print("brief_text not in memory, and bad file missing:", bad_out)

CWD = /home/epistachio/workspace/news-briefing
feeds.csv copied -> /home/epistachio/workspace/news-briefing/configs/feeds.csv
briefing saved(from brief_text) -> /home/epistachio/workspace/news-briefing/data/processed/politics_briefing.md


In [26]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import re
from html import unescape
from datetime import datetime, timedelta, timezone

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN

# =========================
# 0) 경로 고정 (표준 루트)
# =========================
ROOT = Path.home() / "workspace" / "news-briefing"
DB_PATH = ROOT / "db" / "news.db"
FEEDS_CSV = ROOT / "configs" / "feeds.csv"

assert DB_PATH.exists(), f"DB not found: {DB_PATH}"
assert FEEDS_CSV.exists(), f"feeds.csv not found: {FEEDS_CSV}"

feeds_df = pd.read_csv(FEEDS_CSV)
feed_to_section = dict(zip(feeds_df["feed_name"], feeds_df["section"]))
feed_to_bucket  = dict(zip(feeds_df["feed_name"], feeds_df.get("politics_bucket", pd.Series([""]*len(feeds_df))).fillna("")))

# =========================
# 1) 최근 24시간 로드
# =========================
HOURS = 24
since = (datetime.now(timezone.utc) - timedelta(hours=HOURS)).isoformat()

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query(
    """
    SELECT item_id, feed_name, title, summary, link, published, fetched_at
    FROM items
    WHERE fetched_at >= ?
    """,
    conn,
    params=[since]
)

df["section"] = df["feed_name"].map(feed_to_section)
df["politics_bucket"] = df["feed_name"].map(feed_to_bucket).fillna("")

# =========================
# 2) 텍스트 클린
# =========================
TAG_RE = re.compile(r"<[^>]+>")
def clean_text(x):
    if not isinstance(x, str):
        return ""
    x = unescape(x)
    x = TAG_RE.sub(" ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

df["title_clean"] = df["title"].map(clean_text)
df["summary_clean"] = df["summary"].map(clean_text)
df["text"] = (df["title_clean"] + " " + df["summary_clean"]).str.strip()
df = df[df["text"].str.len() >= 15].copy()

# =========================
# 3) 클러스터링 유틸
# =========================
def cluster_df(dfx: pd.DataFrame, eps: float, min_samples: int = 2, min_df: int = 2):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=min_df)
    X = vec.fit_transform(dfx["text"])
    labels = DBSCAN(eps=eps, min_samples=min_samples, metric="cosine").fit_predict(X)
    dfo = dfx.copy()
    dfo["cluster"] = labels
    return dfo, X

def rep_index_by_centroid(dfo: pd.DataFrame, X):
    # 클러스터별 centroid와 가장 유사한 문서 1개를 대표로
    idx_to_pos = {idx: pos for pos, idx in enumerate(dfo.index)}
    rows = []
    core = dfo[dfo["cluster"] != -1]
    for c, g in core.groupby("cluster"):
        idxs = g.index.tolist()
        Xc = X[[idx_to_pos[i] for i in idxs], :]
        centroid = Xc.mean(axis=0)
        sims = (Xc @ centroid.T).A.ravel()
        rep_idx = idxs[int(np.argmax(sims))]
        top3 = g["title_clean"].head(3).tolist()
        rows.append({
            "cluster": int(c),
            "size": int(len(g)),
            "sources": int(g["feed_name"].nunique()),
            "representative_title": dfo.loc[rep_idx, "title_clean"],
            "representative_link": dfo.loc[rep_idx, "link"],
            "top3_titles": " / ".join(top3),
        })
    out = pd.DataFrame(rows).sort_values(["size","sources","cluster"], ascending=[False, False, True]).reset_index(drop=True)
    return out

def rep_by_bucket(dfo: pd.DataFrame, X, buckets=("conservative","progressive","centrist")):
    idx_to_pos = {idx: pos for pos, idx in enumerate(dfo.index)}
    rows = []
    core = dfo[dfo["cluster"] != -1]
    for c, g in core.groupby("cluster"):
        idxs = g.index.tolist()
        Xc = X[[idx_to_pos[i] for i in idxs], :]
        centroid = Xc.mean(axis=0)

        cover = [b for b in buckets if (g["politics_bucket"] == b).any()]
        row = {"cluster": int(c), "size": int(len(g)), "sources": int(g["feed_name"].nunique()), "bucket_cover": len(cover)}

        for b in buckets:
            gb = g[g["politics_bucket"] == b]
            if len(gb) == 0:
                row[f"{b}_title"] = ""
                row[f"{b}_link"] = ""
                continue
            bidxs = gb.index.tolist()
            Xb = X[[idx_to_pos[i] for i in bidxs], :]
            sims = (Xb @ centroid.T).A.ravel()
            rep = bidxs[int(np.argmax(sims))]
            row[f"{b}_title"] = dfo.loc[rep, "title_clean"]
            row[f"{b}_link"] = dfo.loc[rep, "link"]

        rows.append(row)

    out = pd.DataFrame(rows).sort_values(
        ["bucket_cover", "size", "sources", "cluster"],
        ascending=[False, False, False, True]
    ).reset_index(drop=True)
    return out

# =========================
# 4) 섹션별 클러스터링 파라미터
# =========================
EPS_MAP = {
    "politics": 0.70,        # 너가 튜닝한 값
    "economy": 0.65,
    "society": 0.65,
    "international": 0.65,
}

# =========================
# 5) 브리핑 생성
# =========================
bucket_kr = {"conservative":"보수", "progressive":"진보", "centrist":"중도"}
buckets = ("conservative","progressive","centrist")

brief = []
brief.append("# 아침 브리핑")
brief.append(f"- 생성 시각(로컬): {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
brief.append(f"- 범위: 최근 {HOURS}시간 (fetched_at >= {since})")
brief.append(f"- 총 기사 수(클린 후): {len(df)}")
brief.append("")

# 정치: 3관점 비교
pol = df[df["section"] == "politics"].copy()
brief.append("## 정치 (보수/진보/중도 비교)")
if len(pol) == 0:
    brief.append("- (데이터 없음)\n")
else:
    pol2, Xp = cluster_df(pol, eps=EPS_MAP["politics"])
    labels = pol2["cluster"].to_numpy()
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int((labels == -1).sum())
    brief.append(f"- 클러스터: {n_clusters}, 노이즈: {n_noise}\n")

    cmp = rep_by_bucket(pol2, Xp, buckets=buckets)
    TOP_N = 10
    sel = cmp.head(TOP_N)

    for i, row in enumerate(sel.itertuples(index=False), start=1):
        brief.append(f"### 정치 이슈 {i} (기사 {row.size} / 매체 {row.sources} / 관점 {row.bucket_cover}종)")
        for b in buckets:
            title = getattr(row, f"{b}_title")
            link  = getattr(row, f"{b}_link")
            if isinstance(title, str) and title.strip():
                brief.append(f"- [{bucket_kr[b]}] {title} — {link}")
            else:
                brief.append(f"- [{bucket_kr[b]}] (해당 관점 기사 없음)")
        brief.append("")

# 경제/사회/세계: 대표기사 + top3
for sec in ["economy", "society", "international"]:
    dsec = df[df["section"] == sec].copy()
    title_kr = {"economy":"경제", "society":"사회", "international":"세계"}[sec]
    brief.append(f"## {title_kr}")
    if len(dsec) == 0:
        brief.append("- (데이터 없음)\n")
        continue

    dsec2, Xs = cluster_df(dsec, eps=EPS_MAP[sec])
    labels = dsec2["cluster"].to_numpy()
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int((labels == -1).sum())
    brief.append(f"- 클러스터: {n_clusters}, 노이즈: {n_noise}\n")

    summ = rep_index_by_centroid(dsec2, Xs)
    TOP_N = 7
    for i, row in enumerate(summ.head(TOP_N).itertuples(index=False), start=1):
        brief.append(f"### {title_kr} 이슈 {i} (기사 {row.size} / 매체 {row.sources})")
        brief.append(f"- 대표: {row.representative_title} — {row.representative_link}")
        brief.append(f"- 참고(상위 3개 제목): {row.top3_titles}")
        brief.append("")
    brief.append("")

brief_text = "\n".join(brief)

OUT_PATH = ROOT / "data" / "processed" / "morning_briefing.md"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUT_PATH.write_text(brief_text, encoding="utf-8")

print("saved:", OUT_PATH)
print("preview:\n")
print(brief_text[:1200])

saved: /home/epistachio/workspace/news-briefing/data/processed/morning_briefing.md
preview:

# 아침 브리핑
- 생성 시각(로컬): 2026-05-14 18:06:23
- 범위: 최근 24시간 (fetched_at >= 2026-05-13T09:06:23.441683+00:00)
- 총 기사 수(클린 후): 711

## 정치 (보수/진보/중도 비교)
- 클러스터: 25, 노이즈: 19

### 정치 이슈 1 (기사 6 / 매체 2 / 관점 0종)
- [보수] (해당 관점 기사 없음)
- [진보] (해당 관점 기사 없음)
- [중도] (해당 관점 기사 없음)

### 정치 이슈 2 (기사 6 / 매체 2 / 관점 0종)
- [보수] (해당 관점 기사 없음)
- [진보] (해당 관점 기사 없음)
- [중도] (해당 관점 기사 없음)

### 정치 이슈 3 (기사 5 / 매체 2 / 관점 0종)
- [보수] (해당 관점 기사 없음)
- [진보] (해당 관점 기사 없음)
- [중도] (해당 관점 기사 없음)

### 정치 이슈 4 (기사 5 / 매체 2 / 관점 0종)
- [보수] (해당 관점 기사 없음)
- [진보] (해당 관점 기사 없음)
- [중도] (해당 관점 기사 없음)

### 정치 이슈 5 (기사 4 / 매체 2 / 관점 0종)
- [보수] (해당 관점 기사 없음)
- [진보] (해당 관점 기사 없음)
- [중도] (해당 관점 기사 없음)

### 정치 이슈 6 (기사 4 / 매체 2 / 관점 0종)
- [보수] (해당 관점 기사 없음)
- [진보] (해당 관점 기사 없음)
- [중도] (해당 관점 기사 없음)

### 정치 이슈 7 (기사 3 / 매체 2 / 관점 0종)
- [보수] (해당 관점 기사 없음)
- [진보] (해당 관점 기사 없음)
- [중도] (해당 관점 기사 없음)

### 정치 이슈 8 (기사 3 / 매체 2 / 관점 0종)
- [보수] (해당 관점 기사 없음)

In [27]:
from pathlib import Path
import sqlite3
import pandas as pd
import feedparser
from datetime import datetime, timezone
import hashlib
from urllib.parse import urlparse, urlunparse

ROOT = Path.home() / "workspace" / "news-briefing"
FEEDS_CSV = ROOT / "configs" / "feeds.csv"
DB_PATH = ROOT / "db" / "news.db"

feeds_df = pd.read_csv(FEEDS_CSV)

# dtype 안전화(이전에 너가 겪은 float64 문제 방지)
if "politics_bucket" not in feeds_df.columns:
    feeds_df["politics_bucket"] = ""
feeds_df["politics_bucket"] = feeds_df["politics_bucket"].fillna("").astype("string")

# 1) Newsis RSS 추가 (출처: 뉴시스 RSS 페이지에 섹션별 링크가 있음)
# https://www.newsis.com/RSS/ 에서 /RSS/politics.xml 같은 상대경로 제공
new_rows = [
    {"publisher":"newsis", "section":"politics",      "feed_name":"newsis_politics",      "feed_url":"https://www.newsis.com/RSS/politics.xml",       "politics_bucket":"centrist"},
    {"publisher":"newsis", "section":"economy",       "feed_name":"newsis_economy",       "feed_url":"https://www.newsis.com/RSS/economy.xml",        "politics_bucket":""},
    {"publisher":"newsis", "section":"society",       "feed_name":"newsis_society",       "feed_url":"https://www.newsis.com/RSS/society.xml",        "politics_bucket":""},
    {"publisher":"newsis", "section":"international", "feed_name":"newsis_international", "feed_url":"https://www.newsis.com/RSS/international.xml",  "politics_bucket":""},
]
existing = set(feeds_df["feed_name"].astype(str))
add_df = pd.DataFrame([r for r in new_rows if r["feed_name"] not in existing])
if len(add_df):
    add_df["politics_bucket"] = add_df["politics_bucket"].fillna("").astype("string")
    feeds_df = pd.concat([feeds_df, add_df], ignore_index=True)
    feeds_df.to_csv(FEEDS_CSV, index=False, encoding="utf-8-sig")
print("feeds rows:", len(feeds_df))
print("added:", add_df["feed_name"].tolist())

# 2) newsis 피드만 증분 수집 -> SQLite 적재
def normalize_url(u: str | None) -> str | None:
    if not isinstance(u, str) or not u.strip():
        return None
    p = urlparse(u.strip())
    return urlunparse((p.scheme, p.netloc, p.path, "", "", ""))

def make_item_id(feed_url: str | None, guid: str | None, link: str | None, title: str | None, published: str | None) -> str:
    base = guid or normalize_url(link) or f"{title}|{published}|{feed_url}"
    base = (base or "").strip()
    return hashlib.sha256(base.encode("utf-8")).hexdigest()

def fetch_feed(feed_name: str, feed_url: str, limit: int = 50) -> pd.DataFrame:
    d = feedparser.parse(feed_url)
    fetched_at = datetime.now(timezone.utc).isoformat()
    rows = []
    for e in d.entries[:limit]:
        link = getattr(e, "link", None)
        guid = getattr(e, "id", None) or getattr(e, "guid", None)
        title = getattr(e, "title", None)
        published = getattr(e, "published", None)
        summary = getattr(e, "summary", None)
        link_norm = normalize_url(link)
        item_id = make_item_id(feed_url, guid, link_norm, title, published)
        rows.append({
            "item_id": item_id,
            "feed_name": feed_name,
            "feed_url": feed_url,
            "title": title,
            "link": link,
            "link_norm": link_norm,
            "published": published,
            "summary": summary,
            "guid": guid,
            "fetched_at": fetched_at,
        })
    return pd.DataFrame(rows)

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("""
CREATE TABLE IF NOT EXISTS items (
    item_id TEXT PRIMARY KEY,
    feed_name TEXT,
    feed_url TEXT,
    title TEXT,
    link TEXT,
    link_norm TEXT,
    published TEXT,
    summary TEXT,
    guid TEXT,
    fetched_at TEXT
)
""")
conn.commit()

before = cur.execute("SELECT COUNT(*) FROM items").fetchone()[0]

report = []
newsis_feeds = feeds_df[feeds_df["publisher"].astype(str) == "newsis"][["feed_name","feed_url"]]
for _, r in newsis_feeds.iterrows():
    fn, url = str(r["feed_name"]), str(r["feed_url"])
    df = fetch_feed(fn, url, limit=50)
    recs = df[["item_id","feed_name","feed_url","title","link","link_norm","published","summary","guid","fetched_at"]].to_records(index=False)

    cur.executemany("""
    INSERT OR IGNORE INTO items
    (item_id, feed_name, feed_url, title, link, link_norm, published, summary, guid, fetched_at)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, list(recs))
    conn.commit()

    now = cur.execute("SELECT COUNT(*) FROM items").fetchone()[0]
    inserted = now - (before + sum(x["inserted"] for x in report))
    report.append({"feed_name": fn, "fetched": len(df), "inserted": inserted})

after = cur.execute("SELECT COUNT(*) FROM items").fetchone()[0]
print("items before:", before, "after:", after, "delta:", after-before)
pd.DataFrame(report)

feeds rows: 12
added: ['newsis_politics', 'newsis_economy', 'newsis_society', 'newsis_international']
items before: 2114 after: 2114 delta: 0


,feed_name,fetched,inserted
0,newsis_politics,50,0
1,newsis_economy,50,0
2,newsis_society,50,0
3,newsis_international,50,0


In [28]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import re
from html import unescape
from datetime import datetime, timedelta, timezone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN

ROOT = Path.home() / "workspace" / "news-briefing"
DB_PATH = ROOT / "db" / "news.db"
FEEDS_CSV = ROOT / "configs" / "feeds.csv"

feeds_df = pd.read_csv(FEEDS_CSV)
feed_to_section = dict(zip(feeds_df["feed_name"], feeds_df["section"]))
feed_to_bucket  = dict(zip(feeds_df["feed_name"], feeds_df["politics_bucket"].fillna("")))

HOURS = 24
since = (datetime.now(timezone.utc) - timedelta(hours=HOURS)).isoformat()

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("""
SELECT item_id, feed_name, title, summary, link, published, fetched_at
FROM items
WHERE fetched_at >= ?
""", conn, params=[since])

df["section"] = df["feed_name"].map(feed_to_section)
df["politics_bucket"] = df["feed_name"].map(feed_to_bucket).fillna("")

# 정치만
pol = df[df["section"] == "politics"].copy()

TAG_RE = re.compile(r"<[^>]+>")
def clean_text(x):
    if not isinstance(x, str): return ""
    x = unescape(x)
    x = TAG_RE.sub(" ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

pol["title_clean"] = pol["title"].map(clean_text)
pol["summary_clean"] = pol["summary"].map(clean_text)
pol["text"] = (pol["title_clean"] + " " + pol["summary_clean"]).str.strip()
pol = pol[pol["text"].str.len() >= 15].copy()

EPS = 0.70
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2)
X = vec.fit_transform(pol["text"])
labels = DBSCAN(eps=EPS, min_samples=2, metric="cosine").fit_predict(X)
pol["cluster"] = labels

core = pol[pol["cluster"] != -1]
buckets = ["conservative","progressive","centrist"]

cover = core.groupby("cluster")["politics_bucket"].apply(lambda s: len(set([x for x in s if x in buckets])))
dist = cover.value_counts().sort_index()

n_clusters = cover.shape[0]
n_noise = int((labels == -1).sum())

print("EPS =", EPS)
print("rows:", len(pol), "| clusters:", n_clusters, "| noise:", n_noise)
print("bucket_cover distribution (1/2/3):", dist.to_dict())

# 3관점 클러스터만 상위 몇 개 보기
c3 = cover[cover >= 3].index.tolist()
if c3:
    view = (core[core["cluster"].isin(c3)]
            .groupby("cluster")
            .apply(lambda g: g[["politics_bucket","feed_name","title_clean","link"]].head(6))
            .reset_index(level=0)
           )
    display(view.head(18))
else:
    print("3관점 클러스터가 아직 없음 (centrist 추가가 더 필요할 수 있음)")

EPS = 0.7
rows: 144 | clusters: 33 | noise: 43
bucket_cover distribution (1/2/3): {0: 21, 1: 12}
3관점 클러스터가 아직 없음 (centrist 추가가 더 필요할 수 있음)


In [29]:
from pathlib import Path
import sqlite3, re
import pandas as pd
import numpy as np
from html import unescape
from datetime import datetime, timedelta, timezone

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN

ROOT = Path.home() / "workspace" / "news-briefing"
DB_PATH = ROOT / "db" / "news.db"
FEEDS_CSV = ROOT / "configs" / "feeds.csv"

feeds_df = pd.read_csv(FEEDS_CSV)
feed_to_section = dict(zip(feeds_df["feed_name"], feeds_df["section"]))
feed_to_bucket  = dict(zip(feeds_df["feed_name"], feeds_df["politics_bucket"].fillna("")))

# ---- load last 24h ----
HOURS = 24
since = (datetime.now(timezone.utc) - timedelta(hours=HOURS)).isoformat()

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("""
SELECT item_id, feed_name, title, summary, link, fetched_at
FROM items
WHERE fetched_at >= ?
""", conn, params=[since])

df["section"] = df["feed_name"].map(feed_to_section)
df["politics_bucket"] = df["feed_name"].map(feed_to_bucket).fillna("")

pol = df[df["section"] == "politics"].copy()

# ---- clean ----
TAG_RE = re.compile(r"<[^>]+>")
def clean_text(x):
    if not isinstance(x, str): return ""
    x = unescape(x)
    x = TAG_RE.sub(" ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

pol["title_clean"] = pol["title"].map(clean_text)
pol["summary_clean"] = pol["summary"].map(clean_text)
pol["text"] = (pol["title_clean"] + " " + pol["summary_clean"]).str.strip()
pol = pol[pol["text"].str.len() >= 15].copy()

print("politics rows:", len(pol), "| bucket counts:", pol["politics_bucket"].value_counts().to_dict())

# ---- cluster ----
EPS = 0.70
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2)
X = vec.fit_transform(pol["text"])
labels = DBSCAN(eps=EPS, min_samples=2, metric="cosine").fit_predict(X)
pol["cluster"] = labels

core = pol[pol["cluster"] != -1].copy()
print("clusters:", core["cluster"].nunique(), "| noise:", int((labels == -1).sum()))

# ---- fill buckets from global pool ----
buckets = ["conservative","progressive","centrist"]

idx_to_pos = {idx: pos for pos, idx in enumerate(pol.index)}

bucket_candidates = {b: pol[pol["politics_bucket"] == b].index.tolist() for b in buckets}

SIM_MIN = 0.18
used = {b:set() for b in buckets}

rows = []
for c, g in core.groupby("cluster"):
    idxs = g.index.tolist()
    Xc = X[[idx_to_pos[i] for i in idxs], :]

    # ✅ FIX: centroid를 1D ndarray로 변환 + 수동 L2 정규화
    centroid = np.asarray(Xc.mean(axis=0)).ravel()
    centroid = centroid / (np.linalg.norm(centroid) + 1e-12)

    row = {"cluster": int(c), "size": int(len(g)), "sources": int(g["feed_name"].nunique())}

    filled = 0
    for b in buckets:
        cand = [i for i in bucket_candidates[b] if i not in used[b]]
        if not cand:
            row[f"{b}_title"] = ""
            row[f"{b}_link"] = ""
            row[f"{b}_sim"] = 0.0
            continue

        Xb = X[[idx_to_pos[i] for i in cand], :]

        # ✅ FIX: sparse.dot(1D ndarray) -> (n,) ndarray
        sims = Xb.dot(centroid)
        best_pos = int(np.argmax(sims))
        best_sim = float(sims[best_pos])
        best_idx = cand[best_pos]

        if best_sim < SIM_MIN:
            row[f"{b}_title"] = ""
            row[f"{b}_link"] = ""
            row[f"{b}_sim"] = best_sim
        else:
            row[f"{b}_title"] = pol.loc[best_idx, "title_clean"]
            row[f"{b}_link"]  = pol.loc[best_idx, "link"]
            row[f"{b}_sim"]   = best_sim
            used[b].add(best_idx)
            filled += 1

    row["bucket_filled"] = filled
    rows.append(row)

cmp_fill = (pd.DataFrame(rows)
            .sort_values(["bucket_filled","size","sources","cluster"], ascending=[False, False, False, True])
            .reset_index(drop=True))

print("SIM_MIN =", SIM_MIN)
print("filled distribution:", cmp_fill["bucket_filled"].value_counts().sort_index().to_dict())

# 3관점이 채워진 이슈 상위 10개만
display(cmp_fill[cmp_fill["bucket_filled"]==3].head(10)[
    ["cluster","size","sources",
     "conservative_title","conservative_sim",
     "progressive_title","progressive_sim",
     "centrist_title","centrist_sim"]
])

politics rows: 144 | bucket counts: {'': 94, 'centrist': 50}
clusters: 33 | noise: 43
SIM_MIN = 0.18
filled distribution: {0: 16, 1: 17}


,cluster,size,sources,conservative_title,conservative_sim,progressive_title,progressive_sim,centrist_title,centrist_sim


In [30]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import re
from html import unescape
from datetime import datetime, timedelta, timezone

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN

# =========================
# 0) 경로 고정
# =========================
ROOT = Path.home() / "workspace" / "news-briefing"
DB_PATH = ROOT / "db" / "news.db"
FEEDS_CSV = ROOT / "configs" / "feeds.csv"

feeds_df = pd.read_csv(FEEDS_CSV)
feed_to_section = dict(zip(feeds_df["feed_name"], feeds_df["section"]))
feed_to_bucket  = dict(zip(feeds_df["feed_name"], feeds_df["politics_bucket"].fillna("")))

# =========================
# 1) 최근 24h 로드
# =========================
HOURS = 24
since = (datetime.now(timezone.utc) - timedelta(hours=HOURS)).isoformat()

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("""
SELECT item_id, feed_name, title, summary, link, fetched_at
FROM items
WHERE fetched_at >= ?
""", conn, params=[since])

df["section"] = df["feed_name"].map(feed_to_section)
df["politics_bucket"] = df["feed_name"].map(feed_to_bucket).fillna("")

# =========================
# 2) 텍스트 클린
# =========================
TAG_RE = re.compile(r"<[^>]+>")
def clean_text(x):
    if not isinstance(x, str):
        return ""
    x = unescape(x)
    x = TAG_RE.sub(" ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

df["title_clean"] = df["title"].map(clean_text)
df["summary_clean"] = df["summary"].map(clean_text)
df["text"] = (df["title_clean"] + " " + df["summary_clean"]).str.strip()
df = df[df["text"].str.len() >= 15].copy()

# =========================
# 3) 클러스터링 유틸
# =========================
def cluster_df(dfx: pd.DataFrame, eps: float, min_samples: int = 2, min_df: int = 2):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=min_df)
    X = vec.fit_transform(dfx["text"])
    labels = DBSCAN(eps=eps, min_samples=min_samples, metric="cosine").fit_predict(X)
    dfo = dfx.copy()
    dfo["cluster"] = labels
    return dfo, X

def rep_index_by_centroid(dfo: pd.DataFrame, X):
    idx_to_pos = {idx: pos for pos, idx in enumerate(dfo.index)}
    rows = []
    core = dfo[dfo["cluster"] != -1]
    for c, g in core.groupby("cluster"):
        idxs = g.index.tolist()
        Xc = X[[idx_to_pos[i] for i in idxs], :]
        centroid = np.asarray(Xc.mean(axis=0)).ravel()
        centroid = centroid / (np.linalg.norm(centroid) + 1e-12)

        sims = Xc.dot(centroid)
        rep_idx = idxs[int(np.argmax(sims))]

        top3 = g["title_clean"].head(3).tolist()
        rows.append({
            "cluster": int(c),
            "size": int(len(g)),
            "sources": int(g["feed_name"].nunique()),
            "representative_title": dfo.loc[rep_idx, "title_clean"],
            "representative_link": dfo.loc[rep_idx, "link"],
            "top3_titles": " / ".join(top3),
        })
    out = pd.DataFrame(rows).sort_values(["size","sources","cluster"], ascending=[False, False, True]).reset_index(drop=True)
    return out

# =========================
# 4) 정치: 3관점 “끌어오기” 대표선정
# =========================
def rep_politics_fill_3views(pol_df: pd.DataFrame, X, sim_min: float = 0.18):
    buckets = ["conservative","progressive","centrist"]
    idx_to_pos = {idx: pos for pos, idx in enumerate(pol_df.index)}

    # bucket별 후보(정치 전체 풀)
    bucket_candidates = {b: pol_df[pol_df["politics_bucket"] == b].index.tolist() for b in buckets}
    used = {b:set() for b in buckets}

    core = pol_df[pol_df["cluster"] != -1]
    rows = []

    for c, g in core.groupby("cluster"):
        idxs = g.index.tolist()
        Xc = X[[idx_to_pos[i] for i in idxs], :]

        centroid = np.asarray(Xc.mean(axis=0)).ravel()
        centroid = centroid / (np.linalg.norm(centroid) + 1e-12)

        row = {
            "cluster": int(c),
            "size": int(len(g)),
            "sources": int(g["feed_name"].nunique()),
        }

        filled = 0
        for b in buckets:
            cand = [i for i in bucket_candidates[b] if i not in used[b]]
            if not cand:
                row[f"{b}_title"] = ""
                row[f"{b}_link"] = ""
                row[f"{b}_sim"] = 0.0
                continue

            Xb = X[[idx_to_pos[i] for i in cand], :]
            sims = Xb.dot(centroid)          # (n,)
            best_pos = int(np.argmax(sims))
            best_sim = float(sims[best_pos])
            best_idx = cand[best_pos]

            if best_sim < sim_min:
                row[f"{b}_title"] = ""
                row[f"{b}_link"] = ""
                row[f"{b}_sim"] = best_sim
            else:
                row[f"{b}_title"] = pol_df.loc[best_idx, "title_clean"]
                row[f"{b}_link"]  = pol_df.loc[best_idx, "link"]
                row[f"{b}_sim"]   = best_sim
                used[b].add(best_idx)
                filled += 1

        row["bucket_filled"] = filled
        rows.append(row)

    out = (pd.DataFrame(rows)
           .sort_values(["bucket_filled","size","sources","cluster"], ascending=[False, False, False, True])
           .reset_index(drop=True))
    return out

# =========================
# 5) 섹션별 파라미터
# =========================
EPS_MAP = {"politics": 0.70, "economy": 0.65, "society": 0.65, "international": 0.65}
SIM_MIN = 0.18
LOW_SIM = 0.25  # 이보다 낮으면 "유사도 낮음" 표시

bucket_kr = {"conservative":"보수", "progressive":"진보", "centrist":"중도"}
buckets = ["conservative","progressive","centrist"]

# =========================
# 6) 브리핑 생성
# =========================
brief = []
brief.append("# 아침 브리핑")
brief.append(f"- 생성 시각(로컬): {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
brief.append(f"- 범위: 최근 {HOURS}시간 (fetched_at >= {since})")
brief.append(f"- 총 기사 수(클린 후): {len(df)}")
brief.append("")

# ---- 정치 ----
pol = df[df["section"] == "politics"].copy()
brief.append("## 정치 (보수/진보/중도 비교)")
if len(pol) == 0:
    brief.append("- (데이터 없음)\n")
else:
    pol2, Xp = cluster_df(pol, eps=EPS_MAP["politics"])
    labels = pol2["cluster"].to_numpy()
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int((labels == -1).sum())
    brief.append(f"- 클러스터: {n_clusters}, 노이즈: {n_noise}")
    brief.append(f"- 3관점 보강: centroid 유사도 매칭(SIM_MIN={SIM_MIN})\n")

    cmp = rep_politics_fill_3views(pol2, Xp, sim_min=SIM_MIN)
    dist = cmp["bucket_filled"].value_counts().sort_index().to_dict()
    brief.append(f"- 관점 채움 분포(bucket_filled=1/2/3): {dist}\n")

    TOP_N = 10
    for i, row in enumerate(cmp.head(TOP_N).itertuples(index=False), start=1):
        brief.append(f"### 정치 이슈 {i} (기사 {row.size} / 매체 {row.sources} / 관점 {row.bucket_filled}종)")
        for b in buckets:
            title = getattr(row, f"{b}_title")
            link  = getattr(row, f"{b}_link")
            sim   = float(getattr(row, f"{b}_sim"))
            if isinstance(title, str) and title.strip():
                flag = " (유사도 낮음)" if sim < LOW_SIM else ""
                brief.append(f"- [{bucket_kr[b]}]{flag} {title} — {link}")
            else:
                brief.append(f"- [{bucket_kr[b]}] (해당 관점 기사 없음)")
        brief.append("")

# ---- 경제/사회/세계 ----
for sec in ["economy", "society", "international"]:
    dsec = df[df["section"] == sec].copy()
    title_kr = {"economy":"경제", "society":"사회", "international":"세계"}[sec]
    brief.append(f"## {title_kr}")
    if len(dsec) == 0:
        brief.append("- (데이터 없음)\n")
        continue

    dsec2, Xs = cluster_df(dsec, eps=EPS_MAP[sec])
    labels = dsec2["cluster"].to_numpy()
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int((labels == -1).sum())
    brief.append(f"- 클러스터: {n_clusters}, 노이즈: {n_noise}\n")

    summ = rep_index_by_centroid(dsec2, Xs)
    TOP_N = 7
    for i, row in enumerate(summ.head(TOP_N).itertuples(index=False), start=1):
        brief.append(f"### {title_kr} 이슈 {i} (기사 {row.size} / 매체 {row.sources})")
        brief.append(f"- 대표: {row.representative_title} — {row.representative_link}")
        brief.append(f"- 참고(상위 3개 제목): {row.top3_titles}")
        brief.append("")
    brief.append("")

brief_text = "\n".join(brief)

OUT_PATH = ROOT / "data" / "processed" / "morning_briefing.md"
OUT_PATH.write_text(brief_text, encoding="utf-8")
print("saved:", OUT_PATH)
print("\npreview:\n")
print(brief_text[:1400])

saved: /home/epistachio/workspace/news-briefing/data/processed/morning_briefing.md

preview:

# 아침 브리핑
- 생성 시각(로컬): 2026-05-14 18:06:24
- 범위: 최근 24시간 (fetched_at >= 2026-05-13T09:06:24.874831+00:00)
- 총 기사 수(클린 후): 711

## 정치 (보수/진보/중도 비교)
- 클러스터: 33, 노이즈: 43
- 3관점 보강: centroid 유사도 매칭(SIM_MIN=0.18)

- 관점 채움 분포(bucket_filled=1/2/3): {0: 16, 1: 17}

### 정치 이슈 1 (기사 15 / 매체 3 / 관점 1종)
- [보수] (해당 관점 기사 없음)
- [진보] (해당 관점 기사 없음)
- [중도] [6·3경기북부]시장·군수 후보 등록 잇따라…본격 선거전(종합) — https://www.newsis.com/view/NISX20260514_0003630156

### 정치 이슈 2 (기사 7 / 매체 3 / 관점 1종)
- [보수] (해당 관점 기사 없음)
- [진보] (해당 관점 기사 없음)
- [중도] 이재명 대통령, 새마을운동중앙회 찾아 현장간담회 [뉴시스Pic] — https://www.newsis.com/view/NISX20260514_0003629866

### 정치 이슈 3 (기사 4 / 매체 2 / 관점 1종)
- [보수] (해당 관점 기사 없음)
- [진보] (해당 관점 기사 없음)
- [중도] 정부 "나무호 피격 대응 조치, 조사 기초해 여타국 대응 등도 종합적 감안" — https://www.newsis.com/view/NISX20260514_0003629974

### 정치 이슈 4 (기사 3 / 매체 2 / 관점 1종)
- [보수] (해당 관점 기사 없음)
- [진보] (해당 관점 기사 없음)
- [중도] 여야 원내대표 상견례…20일 '의장 선출' 본회의 일정 합의 불발 

In [31]:
from pathlib import Path
import sqlite3, re
import pandas as pd
import numpy as np
from html import unescape
from datetime import datetime, timedelta, timezone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN

ROOT = Path.home() / "workspace" / "news-briefing"
DB_PATH = ROOT / "db" / "news.db"
FEEDS_CSV = ROOT / "configs" / "feeds.csv"

feeds_df = pd.read_csv(FEEDS_CSV)
feed_to_section = dict(zip(feeds_df["feed_name"], feeds_df["section"]))
feed_to_bucket  = dict(zip(feeds_df["feed_name"], feeds_df["politics_bucket"].fillna("")))

HOURS = 24
since = (datetime.now(timezone.utc) - timedelta(hours=HOURS)).isoformat()

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("""
SELECT item_id, feed_name, title, summary, link, fetched_at
FROM items
WHERE fetched_at >= ?
""", conn, params=[since])

df["section"] = df["feed_name"].map(feed_to_section)
df["politics_bucket"] = df["feed_name"].map(feed_to_bucket).fillna("")

TAG_RE = re.compile(r"<[^>]+>")
def clean_text(x):
    if not isinstance(x, str): return ""
    x = unescape(x)
    x = TAG_RE.sub(" ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

df["title_clean"] = df["title"].map(clean_text)
df["summary_clean"] = df["summary"].map(clean_text)
df["text"] = (df["title_clean"] + " " + df["summary_clean"]).str.strip()
df = df[df["text"].str.len() >= 15].copy()

# ✅ 출력용 제목(절대 빈 줄 방지)
def make_display_title(row):
    t = row["title_clean"]
    if isinstance(t, str) and t.strip():
        return t.strip()
    t2 = clean_text(row.get("title", ""))
    if isinstance(t2, str) and t2.strip():
        return t2.strip()
    s = row["summary_clean"]
    if isinstance(s, str) and s.strip():
        return (s.strip()[:80] + "…") if len(s.strip()) > 80 else s.strip()
    return "(제목 없음)"
df["display_title"] = df.apply(make_display_title, axis=1)

# ---- 정치 클러스터링 ----
pol = df[df["section"] == "politics"].copy()

EPS = 0.70
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2)
X = vec.fit_transform(pol["text"])
labels = DBSCAN(eps=EPS, min_samples=2, metric="cosine").fit_predict(X)
pol["cluster"] = labels

buckets = ["conservative","progressive","centrist"]
bucket_kr = {"conservative":"보수", "progressive":"진보", "centrist":"중도"}

idx_to_pos = {idx: pos for pos, idx in enumerate(pol.index)}
bucket_candidates_global = {b: pol[pol["politics_bucket"] == b].index.tolist() for b in buckets}

SIM_MIN = 0.18
LOW_SIM = 0.25

used = {b:set() for b in buckets}
rows = []
core = pol[pol["cluster"] != -1]

for c, g in core.groupby("cluster"):
    idxs = g.index.tolist()
    Xc = X[[idx_to_pos[i] for i in idxs], :]
    centroid = np.asarray(Xc.mean(axis=0)).ravel()
    centroid = centroid / (np.linalg.norm(centroid) + 1e-12)

    row = {"cluster": int(c), "size": int(len(g)), "sources": int(g["feed_name"].nunique())}
    filled = 0

    for b in buckets:
        # ✅ 1) 클러스터 내부에 있으면 내부에서 우선 선택
        cand_in = [i for i in g[g["politics_bucket"] == b].index.tolist() if i not in used[b]]
        cand = cand_in

        # ✅ 2) 없으면 전체 풀에서 끌어오기
        if not cand:
            cand = [i for i in bucket_candidates_global[b] if i not in used[b]]

        if not cand:
            row[f"{b}_title"] = ""
            row[f"{b}_link"] = ""
            row[f"{b}_sim"] = 0.0
            continue

        Xb = X[[idx_to_pos[i] for i in cand], :]
        sims = Xb.dot(centroid)
        best_pos = int(np.argmax(sims))
        best_sim = float(sims[best_pos])
        best_idx = cand[best_pos]

        if best_sim < SIM_MIN:
            row[f"{b}_title"] = ""
            row[f"{b}_link"] = ""
            row[f"{b}_sim"] = best_sim
        else:
            row[f"{b}_title"] = pol.loc[best_idx, "display_title"]   # ✅ 빈 줄 방지
            row[f"{b}_link"]  = pol.loc[best_idx, "link"] or ""
            row[f"{b}_sim"]   = best_sim
            used[b].add(best_idx)
            filled += 1

    row["bucket_filled"] = filled
    rows.append(row)

cmp = (pd.DataFrame(rows)
       .sort_values(["bucket_filled","size","sources","cluster"], ascending=[False, False, False, True])
       .reset_index(drop=True))

# ---- 정치 섹션만 뽑아서 미리보기 ----
print("filled distribution:", cmp["bucket_filled"].value_counts().sort_index().to_dict())
for i, r in enumerate(cmp.head(3).itertuples(index=False), start=1):
    print(f"\n### 정치 이슈 {i} (기사 {r.size} / 매체 {r.sources} / 관점 {r.bucket_filled}종)")
    for b in buckets:
        title = getattr(r, f"{b}_title")
        link  = getattr(r, f"{b}_link")
        sim   = float(getattr(r, f"{b}_sim"))
        if isinstance(title, str) and title.strip():
            flag = " (유사도 낮음)" if sim < LOW_SIM else ""
            print(f"- [{bucket_kr[b]}]{flag} {title} — {link}")
        else:
            print(f"- [{bucket_kr[b]}] (해당 관점 기사 없음)")

# ---- 파일 저장은 기존 morning_briefing.md를 그대로 쓰되, 너가 원하면 내가 다음 단계에서 전체 파일 생성까지 합쳐줄게 ----

filled distribution: {0: 16, 1: 17}

### 정치 이슈 1 (기사 15 / 매체 3 / 관점 1종)
- [보수] (해당 관점 기사 없음)
- [진보] (해당 관점 기사 없음)
- [중도] [6·3경기북부]시장·군수 후보 등록 잇따라…본격 선거전(종합) — https://www.newsis.com/view/NISX20260514_0003630156

### 정치 이슈 2 (기사 7 / 매체 3 / 관점 1종)
- [보수] (해당 관점 기사 없음)
- [진보] (해당 관점 기사 없음)
- [중도] 이재명 대통령, 새마을운동중앙회 찾아 현장간담회 [뉴시스Pic] — https://www.newsis.com/view/NISX20260514_0003629866

### 정치 이슈 3 (기사 4 / 매체 2 / 관점 1종)
- [보수] (해당 관점 기사 없음)
- [진보] (해당 관점 기사 없음)
- [중도] 정부 "나무호 피격 대응 조치, 조사 기초해 여타국 대응 등도 종합적 감안" — https://www.newsis.com/view/NISX20260514_0003629974


In [32]:
from pathlib import Path
import sqlite3, re
import pandas as pd
import numpy as np
from html import unescape
from datetime import datetime, timedelta, timezone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN

# ----------------
# Fixed paths
# ----------------
ROOT = Path.home() / "workspace" / "news-briefing"
DB_PATH = ROOT / "db" / "news.db"
FEEDS_CSV = ROOT / "configs" / "feeds.csv"

feeds_df = pd.read_csv(FEEDS_CSV)
feed_to_section = dict(zip(feeds_df["feed_name"], feeds_df["section"]))
feed_to_bucket  = dict(zip(feeds_df["feed_name"], feeds_df["politics_bucket"].fillna("")))

# ----------------
# Load last 24h
# ----------------
HOURS = 24
since = (datetime.now(timezone.utc) - timedelta(hours=HOURS)).isoformat()

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("""
SELECT item_id, feed_name, title, summary, link, fetched_at
FROM items
WHERE fetched_at >= ?
""", conn, params=[since])

df["section"] = df["feed_name"].map(feed_to_section)
df["politics_bucket"] = df["feed_name"].map(feed_to_bucket).fillna("")

# ----------------
# Clean
# ----------------
TAG_RE = re.compile(r"<[^>]+>")
def clean_text(x):
    if not isinstance(x, str): return ""
    x = unescape(x)
    x = TAG_RE.sub(" ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

df["title_clean"] = df["title"].map(clean_text)
df["summary_clean"] = df["summary"].map(clean_text)
df["text"] = (df["title_clean"] + " " + df["summary_clean"]).str.strip()
df = df[df["text"].str.len() >= 15].copy()

def make_display_title(row):
    t = row["title_clean"]
    if isinstance(t, str) and t.strip():
        return t.strip()
    t2 = clean_text(row.get("title", ""))
    if isinstance(t2, str) and t2.strip():
        return t2.strip()
    s = row["summary_clean"]
    if isinstance(s, str) and s.strip():
        s = s.strip()
        return (s[:80] + "…") if len(s) > 80 else s
    return "(제목 없음)"

df["display_title"] = df.apply(make_display_title, axis=1)

# ----------------
# Clustering utils
# ----------------
def cluster_df(dfx: pd.DataFrame, eps: float, min_samples: int = 2, min_df: int = 2):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=min_df)
    X = vec.fit_transform(dfx["text"])
    labels = DBSCAN(eps=eps, min_samples=min_samples, metric="cosine").fit_predict(X)
    dfo = dfx.copy()
    dfo["cluster"] = labels
    return dfo, X

def rep_index_by_centroid(dfo: pd.DataFrame, X):
    idx_to_pos = {idx: pos for pos, idx in enumerate(dfo.index)}
    rows = []
    core = dfo[dfo["cluster"] != -1]
    for c, g in core.groupby("cluster"):
        idxs = g.index.tolist()
        Xc = X[[idx_to_pos[i] for i in idxs], :]
        centroid = np.asarray(Xc.mean(axis=0)).ravel()
        centroid = centroid / (np.linalg.norm(centroid) + 1e-12)

        sims = Xc.dot(centroid)
        rep_idx = idxs[int(np.argmax(sims))]

        top3 = g["display_title"].head(3).tolist()
        rows.append({
            "cluster": int(c),
            "size": int(len(g)),
            "sources": int(g["feed_name"].nunique()),
            "representative_title": dfo.loc[rep_idx, "display_title"],
            "representative_link": dfo.loc[rep_idx, "link"],
            "top3_titles": " / ".join(top3),
        })
    out = pd.DataFrame(rows).sort_values(["size","sources","cluster"], ascending=[False, False, True]).reset_index(drop=True)
    return out

def rep_politics_fill(pol_df: pd.DataFrame, X, sim_min: float, low_sim: float = 0.25):
    buckets = ["conservative","progressive","centrist"]
    idx_to_pos = {idx: pos for pos, idx in enumerate(pol_df.index)}
    bucket_candidates_global = {b: pol_df[pol_df["politics_bucket"] == b].index.tolist() for b in buckets}

    used = {b:set() for b in buckets}
    rows = []
    core = pol_df[pol_df["cluster"] != -1]

    low_sim_count = 0

    for c, g in core.groupby("cluster"):
        idxs = g.index.tolist()
        Xc = X[[idx_to_pos[i] for i in idxs], :]
        centroid = np.asarray(Xc.mean(axis=0)).ravel()
        centroid = centroid / (np.linalg.norm(centroid) + 1e-12)

        row = {"cluster": int(c), "size": int(len(g)), "sources": int(g["feed_name"].nunique())}
        filled = 0

        for b in buckets:
            # 1) cluster 내부 우선
            cand_in = [i for i in g[g["politics_bucket"] == b].index.tolist() if i not in used[b]]
            cand = cand_in

            # 2) 없으면 global에서 끌어오기
            if not cand:
                cand = [i for i in bucket_candidates_global[b] if i not in used[b]]

            if not cand:
                row[f"{b}_title"] = ""
                row[f"{b}_link"] = ""
                row[f"{b}_sim"] = 0.0
                continue

            Xb = X[[idx_to_pos[i] for i in cand], :]
            sims = Xb.dot(centroid)
            best_pos = int(np.argmax(sims))
            best_sim = float(sims[best_pos])
            best_idx = cand[best_pos]

            if best_sim < sim_min:
                row[f"{b}_title"] = ""
                row[f"{b}_link"] = ""
                row[f"{b}_sim"] = best_sim
            else:
                row[f"{b}_title"] = pol_df.loc[best_idx, "display_title"]
                row[f"{b}_link"]  = pol_df.loc[best_idx, "link"] or ""
                row[f"{b}_sim"]   = best_sim
                used[b].add(best_idx)
                filled += 1
                if best_sim < low_sim:
                    low_sim_count += 1

        row["bucket_filled"] = filled
        rows.append(row)

    out = (pd.DataFrame(rows)
           .sort_values(["bucket_filled","size","sources","cluster"], ascending=[False, False, False, True])
           .reset_index(drop=True))

    return out, low_sim_count

# ----------------
# Parameters
# ----------------
EPS_MAP = {"politics": 0.70, "economy": 0.65, "society": 0.65, "international": 0.65}
LOW_SIM = 0.25

# ----------------
# Politics: tune SIM_MIN
# ----------------
pol = df[df["section"] == "politics"].copy()
pol2, Xp = cluster_df(pol, eps=EPS_MAP["politics"])

cands = [0.18, 0.20, 0.22, 0.24, 0.26]
rows = []
best = None

for sm in cands:
    cmp, low_cnt = rep_politics_fill(pol2, Xp, sim_min=sm, low_sim=LOW_SIM)
    dist = cmp["bucket_filled"].value_counts().sort_index().to_dict()
    n3 = dist.get(3, 0)
    n2 = dist.get(2, 0)
    n1 = dist.get(1, 0)

    # 스코어: 3관점 크게 보상, 저유사 매칭은 페널티
    score = (5*n3 + 2*n2) - (1*low_cnt)

    rows.append({
        "SIM_MIN": sm,
        "filled3": n3,
        "filled2": n2,
        "filled1": n1,
        "low_sim_hits": low_cnt,
        "score": score
    })

sweep = pd.DataFrame(rows).sort_values(["score","filled3","filled2","SIM_MIN"], ascending=[False, False, False, True]).reset_index(drop=True)
display(sweep)

BEST_SIM_MIN = float(sweep.loc[0, "SIM_MIN"])
print("BEST_SIM_MIN =", BEST_SIM_MIN)

cmp_best, low_cnt_best = rep_politics_fill(pol2, Xp, sim_min=BEST_SIM_MIN, low_sim=LOW_SIM)
dist_best = cmp_best["bucket_filled"].value_counts().sort_index().to_dict()

# ----------------
# Build briefing
# ----------------
bucket_kr = {"conservative":"보수", "progressive":"진보", "centrist":"중도"}
buckets = ["conservative","progressive","centrist"]

brief = []
brief.append("# 아침 브리핑")
brief.append(f"- 생성 시각(로컬): {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
brief.append(f"- 범위: 최근 {HOURS}시간 (fetched_at >= {since})")
brief.append(f"- 총 기사 수(클린 후): {len(df)}")
brief.append("")

# Politics section
labels = pol2["cluster"].to_numpy()
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = int((labels == -1).sum())

brief.append("## 정치 (보수/진보/중도 비교)")
brief.append(f"- 클러스터: {n_clusters}, 노이즈: {n_noise}")
brief.append(f"- 3관점 보강: centroid 유사도 매칭(BEST_SIM_MIN={BEST_SIM_MIN}, LOW_SIM<{LOW_SIM})")
brief.append(f"- 관점 채움 분포(bucket_filled=1/2/3): {dist_best}\n")

TOP_N = 10
for i, row in enumerate(cmp_best.head(TOP_N).itertuples(index=False), start=1):
    brief.append(f"### 정치 이슈 {i} (기사 {row.size} / 매체 {row.sources} / 관점 {row.bucket_filled}종)")
    for b in buckets:
        title = getattr(row, f"{b}_title")
        link  = getattr(row, f"{b}_link")
        sim   = float(getattr(row, f"{b}_sim"))
        if isinstance(title, str) and title.strip():
            flag = " (유사도 낮음)" if sim < LOW_SIM else ""
            brief.append(f"- [{bucket_kr[b]}]{flag} {title} — {link}")
        else:
            brief.append(f"- [{bucket_kr[b]}] (해당 관점 기사 없음)")
    brief.append("")

# Other sections
for sec in ["economy", "society", "international"]:
    dsec = df[df["section"] == sec].copy()
    title_kr = {"economy":"경제", "society":"사회", "international":"세계"}[sec]
    brief.append(f"## {title_kr}")
    if len(dsec) == 0:
        brief.append("- (데이터 없음)\n")
        continue

    dsec2, Xs = cluster_df(dsec, eps=EPS_MAP[sec])
    labels = dsec2["cluster"].to_numpy()
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int((labels == -1).sum())
    brief.append(f"- 클러스터: {n_clusters}, 노이즈: {n_noise}\n")

    summ = rep_index_by_centroid(dsec2, Xs)
    TOP_N = 7
    for j, r in enumerate(summ.head(TOP_N).itertuples(index=False), start=1):
        brief.append(f"### {title_kr} 이슈 {j} (기사 {r.size} / 매체 {r.sources})")
        brief.append(f"- 대표: {r.representative_title} — {r.representative_link}")
        brief.append(f"- 참고(상위 3개 제목): {r.top3_titles}")
        brief.append("")
    brief.append("")

brief_text = "\n".join(brief)

OUT_PATH = ROOT / "data" / "processed" / "morning_briefing.md"
OUT_PATH.write_text(brief_text, encoding="utf-8")

print("saved:", OUT_PATH)
print("\npreview (politics 1~3):\n")
# 정치 이슈 1~3만 잘라서 출력
lines = brief_text.splitlines()
start = next(i for i,l in enumerate(lines) if l.startswith("## 정치"))
end = start
cnt = 0
for i in range(start, len(lines)):
    if lines[i].startswith("### 정치 이슈"):
        cnt += 1
    if cnt >= 3 and lines[i].strip()=="":
        end = i+1
        break
print("\n".join(lines[start:end]))

,SIM_MIN,filled3,filled2,filled1,low_sim_hits,score
0,0.24,0,0,14,0,0
1,0.26,0,0,14,0,0
2,0.22,0,0,15,1,-1
3,0.20,0,0,16,2,-2
4,0.18,0,0,17,3,-3


BEST_SIM_MIN = 0.24
saved: /home/epistachio/workspace/news-briefing/data/processed/morning_briefing.md

preview (politics 1~3):

## 정치 (보수/진보/중도 비교)
- 클러스터: 33, 노이즈: 43
- 3관점 보강: centroid 유사도 매칭(BEST_SIM_MIN=0.24, LOW_SIM<0.25)
- 관점 채움 분포(bucket_filled=1/2/3): {0: 19, 1: 14}

### 정치 이슈 1 (기사 15 / 매체 3 / 관점 1종)
- [보수] (해당 관점 기사 없음)
- [진보] (해당 관점 기사 없음)
- [중도] [6·3경기북부]시장·군수 후보 등록 잇따라…본격 선거전(종합) — https://www.newsis.com/view/NISX20260514_0003630156

### 정치 이슈 2 (기사 7 / 매체 3 / 관점 1종)
- [보수] (해당 관점 기사 없음)
- [진보] (해당 관점 기사 없음)
- [중도] 이재명 대통령, 새마을운동중앙회 찾아 현장간담회 [뉴시스Pic] — https://www.newsis.com/view/NISX20260514_0003629866

### 정치 이슈 3 (기사 4 / 매체 2 / 관점 1종)
- [보수] (해당 관점 기사 없음)
- [진보] (해당 관점 기사 없음)
- [중도] 정부 "나무호 피격 대응 조치, 조사 기초해 여타국 대응 등도 종합적 감안" — https://www.newsis.com/view/NISX20260514_0003629974



In [33]:
from pathlib import Path
import json

ROOT = Path.home() / "workspace" / "news-briefing"
CFG_PATH = ROOT / "configs" / "briefing_config.json"

cfg = {
    "hours": 24,
    "top_n_politics": 10,
    "top_n_other": 7,

    "eps": {
        "politics": 0.70,
        "economy": 0.65,
        "society": 0.65,
        "international": 0.65
    },

    # 0.18 = 3관점 최대화(대신 유사도 낮음 일부 발생)
    # 0.20~0.22 = 더 엄격(3관점 감소, 유사도 낮음 감소)
    "sim_min": 0.18,

    # 이 값 미만이면 출력에 "(유사도 낮음)" 표시
    "low_sim": 0.25
}

CFG_PATH.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", CFG_PATH)
print(CFG_PATH.read_text(encoding="utf-8")[:400])

saved: /home/epistachio/workspace/news-briefing/configs/briefing_config.json
{
  "hours": 24,
  "top_n_politics": 10,
  "top_n_other": 7,
  "eps": {
    "politics": 0.7,
    "economy": 0.65,
    "society": 0.65,
    "international": 0.65
  },
  "sim_min": 0.18,
  "low_sim": 0.25
}


In [34]:
from pathlib import Path
import json, sqlite3, re
import pandas as pd
import numpy as np
from html import unescape
from datetime import datetime, timedelta, timezone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN

# ----------------
# Load config
# ----------------
ROOT = Path.home() / "workspace" / "news-briefing"
CFG_PATH = ROOT / "configs" / "briefing_config.json"
cfg = json.loads(CFG_PATH.read_text(encoding="utf-8"))

HOURS = int(cfg["hours"])
TOP_N_POL = int(cfg["top_n_politics"])
TOP_N_OTH = int(cfg["top_n_other"])

EPS_MAP = cfg["eps"]
SIM_MIN = float(cfg["sim_min"])
LOW_SIM = float(cfg["low_sim"])

DB_PATH = ROOT / "db" / "news.db"
FEEDS_CSV = ROOT / "configs" / "feeds.csv"

feeds_df = pd.read_csv(FEEDS_CSV)
feed_to_section = dict(zip(feeds_df["feed_name"], feeds_df["section"]))
feed_to_bucket  = dict(zip(feeds_df["feed_name"], feeds_df["politics_bucket"].fillna("")))

# ----------------
# Load last N hours
# ----------------
since = (datetime.now(timezone.utc) - timedelta(hours=HOURS)).isoformat()
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("""
SELECT item_id, feed_name, title, summary, link, fetched_at
FROM items
WHERE fetched_at >= ?
""", conn, params=[since])

df["section"] = df["feed_name"].map(feed_to_section)
df["politics_bucket"] = df["feed_name"].map(feed_to_bucket).fillna("")

# ----------------
# Clean
# ----------------
TAG_RE = re.compile(r"<[^>]+>")
def clean_text(x):
    if not isinstance(x, str): return ""
    x = unescape(x)
    x = TAG_RE.sub(" ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

df["title_clean"] = df["title"].map(clean_text)
df["summary_clean"] = df["summary"].map(clean_text)
df["text"] = (df["title_clean"] + " " + df["summary_clean"]).str.strip()
df = df[df["text"].str.len() >= 15].copy()

def make_display_title(row):
    t = row["title_clean"]
    if isinstance(t, str) and t.strip():
        return t.strip()
    t2 = clean_text(row.get("title", ""))
    if isinstance(t2, str) and t2.strip():
        return t2.strip()
    s = row["summary_clean"]
    if isinstance(s, str) and s.strip():
        s = s.strip()
        return (s[:80] + "…") if len(s) > 80 else s
    return "(제목 없음)"

df["display_title"] = df.apply(make_display_title, axis=1)

# ----------------
# Utils
# ----------------
def cluster_df(dfx: pd.DataFrame, eps: float, min_samples: int = 2, min_df: int = 2):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=min_df)
    X = vec.fit_transform(dfx["text"])
    labels = DBSCAN(eps=float(eps), min_samples=min_samples, metric="cosine").fit_predict(X)
    dfo = dfx.copy()
    dfo["cluster"] = labels
    return dfo, X

def rep_index_by_centroid(dfo: pd.DataFrame, X):
    idx_to_pos = {idx: pos for pos, idx in enumerate(dfo.index)}
    rows = []
    core = dfo[dfo["cluster"] != -1]
    for c, g in core.groupby("cluster"):
        idxs = g.index.tolist()
        Xc = X[[idx_to_pos[i] for i in idxs], :]
        centroid = np.asarray(Xc.mean(axis=0)).ravel()
        centroid = centroid / (np.linalg.norm(centroid) + 1e-12)
        sims = Xc.dot(centroid)
        rep_idx = idxs[int(np.argmax(sims))]
        top3 = g["display_title"].head(3).tolist()
        rows.append({
            "cluster": int(c),
            "size": int(len(g)),
            "sources": int(g["feed_name"].nunique()),
            "representative_title": dfo.loc[rep_idx, "display_title"],
            "representative_link": dfo.loc[rep_idx, "link"],
            "top3_titles": " / ".join(top3),
        })
    return (pd.DataFrame(rows)
            .sort_values(["size","sources","cluster"], ascending=[False, False, True])
            .reset_index(drop=True))

def rep_politics_fill(pol_df: pd.DataFrame, X, sim_min: float, low_sim: float):
    buckets = ["conservative","progressive","centrist"]
    idx_to_pos = {idx: pos for pos, idx in enumerate(pol_df.index)}
    bucket_candidates_global = {b: pol_df[pol_df["politics_bucket"] == b].index.tolist() for b in buckets}

    used = {b:set() for b in buckets}
    rows = []
    core = pol_df[pol_df["cluster"] != -1]

    for c, g in core.groupby("cluster"):
        idxs = g.index.tolist()
        Xc = X[[idx_to_pos[i] for i in idxs], :]
        centroid = np.asarray(Xc.mean(axis=0)).ravel()
        centroid = centroid / (np.linalg.norm(centroid) + 1e-12)

        row = {"cluster": int(c), "size": int(len(g)), "sources": int(g["feed_name"].nunique())}
        filled = 0

        for b in buckets:
            cand_in = [i for i in g[g["politics_bucket"] == b].index.tolist() if i not in used[b]]
            cand = cand_in if cand_in else [i for i in bucket_candidates_global[b] if i not in used[b]]

            if not cand:
                row[f"{b}_title"] = ""
                row[f"{b}_link"] = ""
                row[f"{b}_sim"] = 0.0
                continue

            Xb = X[[idx_to_pos[i] for i in cand], :]
            sims = Xb.dot(centroid)
            best_pos = int(np.argmax(sims))
            best_sim = float(sims[best_pos])
            best_idx = cand[best_pos]

            if best_sim < sim_min:
                row[f"{b}_title"] = ""
                row[f"{b}_link"] = ""
                row[f"{b}_sim"] = best_sim
            else:
                row[f"{b}_title"] = pol_df.loc[best_idx, "display_title"]
                row[f"{b}_link"]  = pol_df.loc[best_idx, "link"] or ""
                row[f"{b}_sim"]   = best_sim
                used[b].add(best_idx)
                filled += 1

        row["bucket_filled"] = filled
        rows.append(row)

    return (pd.DataFrame(rows)
            .sort_values(["bucket_filled","size","sources","cluster"], ascending=[False, False, False, True])
            .reset_index(drop=True))

# ----------------
# Build briefing
# ----------------
bucket_kr = {"conservative":"보수", "progressive":"진보", "centrist":"중도"}
buckets = ["conservative","progressive","centrist"]

brief = []
brief.append("# 아침 브리핑")
brief.append(f"- 생성 시각(로컬): {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
brief.append(f"- 범위: 최근 {HOURS}시간 (fetched_at >= {since})")
brief.append(f"- 총 기사 수(클린 후): {len(df)}")
brief.append(f"- 설정: eps(politics)={EPS_MAP['politics']}, sim_min={SIM_MIN}, low_sim={LOW_SIM}")
brief.append("")

# Politics
pol = df[df["section"] == "politics"].copy()
brief.append("## 정치 (보수/진보/중도 비교)")
if len(pol) == 0:
    brief.append("- (데이터 없음)\n")
else:
    pol2, Xp = cluster_df(pol, eps=EPS_MAP["politics"])
    labels = pol2["cluster"].to_numpy()
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int((labels == -1).sum())
    brief.append(f"- 클러스터: {n_clusters}, 노이즈: {n_noise}")
    brief.append(f"- 3관점 보강: centroid 유사도 매칭(sim_min={SIM_MIN}, low_sim<{LOW_SIM})\n")

    cmp = rep_politics_fill(pol2, Xp, sim_min=SIM_MIN, low_sim=LOW_SIM)
    dist = cmp["bucket_filled"].value_counts().sort_index().to_dict()
    brief.append(f"- 관점 채움 분포(bucket_filled=1/2/3): {dist}\n")

    for i, row in enumerate(cmp.head(TOP_N_POL).itertuples(index=False), start=1):
        brief.append(f"### 정치 이슈 {i} (기사 {row.size} / 매체 {row.sources} / 관점 {row.bucket_filled}종)")
        for b in buckets:
            title = getattr(row, f"{b}_title")
            link  = getattr(row, f"{b}_link")
            sim   = float(getattr(row, f"{b}_sim"))
            if isinstance(title, str) and title.strip():
                flag = " (유사도 낮음)" if sim < LOW_SIM else ""
                brief.append(f"- [{bucket_kr[b]}]{flag} {title} — {link}")
            else:
                brief.append(f"- [{bucket_kr[b]}] (해당 관점 기사 없음)")
        brief.append("")

# Others
for sec in ["economy", "society", "international"]:
    dsec = df[df["section"] == sec].copy()
    title_kr = {"economy":"경제", "society":"사회", "international":"세계"}[sec]
    brief.append(f"## {title_kr}")
    if len(dsec) == 0:
        brief.append("- (데이터 없음)\n")
        continue

    dsec2, Xs = cluster_df(dsec, eps=EPS_MAP[sec])
    labels = dsec2["cluster"].to_numpy()
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int((labels == -1).sum())
    brief.append(f"- 클러스터: {n_clusters}, 노이즈: {n_noise}\n")

    summ = rep_index_by_centroid(dsec2, Xs)
    for j, r in enumerate(summ.head(TOP_N_OTH).itertuples(index=False), start=1):
        brief.append(f"### {title_kr} 이슈 {j} (기사 {r.size} / 매체 {r.sources})")
        brief.append(f"- 대표: {r.representative_title} — {r.representative_link}")
        brief.append(f"- 참고(상위 3개 제목): {r.top3_titles}")
        brief.append("")
    brief.append("")

brief_text = "\n".join(brief)

OUT_PATH = ROOT / "data" / "processed" / "morning_briefing.md"
OUT_PATH.write_text(brief_text, encoding="utf-8")

print("saved:", OUT_PATH)
print("\npreview (정치 이슈 1):\n")
# 정치 이슈 1만 미리보기
lines = brief_text.splitlines()
start = next(i for i,l in enumerate(lines) if l.startswith("## 정치"))
end = start
seen = 0
for i in range(start, len(lines)):
    if lines[i].startswith("### 정치 이슈"):
        seen += 1
    if seen >= 1 and lines[i].strip()=="":
        end = i+1
        break
print("\n".join(lines[start:end]))

saved: /home/epistachio/workspace/news-briefing/data/processed/morning_briefing.md

preview (정치 이슈 1):

## 정치 (보수/진보/중도 비교)
- 클러스터: 33, 노이즈: 43
- 3관점 보강: centroid 유사도 매칭(sim_min=0.18, low_sim<0.25)

- 관점 채움 분포(bucket_filled=1/2/3): {0: 16, 1: 17}

### 정치 이슈 1 (기사 15 / 매체 3 / 관점 1종)
- [보수] (해당 관점 기사 없음)
- [진보] (해당 관점 기사 없음)
- [중도] [6·3경기북부]시장·군수 후보 등록 잇따라…본격 선거전(종합) — https://www.newsis.com/view/NISX20260514_0003630156

